In [117]:
from pathlib import Pathfrom typing import Optionalimport csvimport pandas as pdimport numpy as npTARGET_YEAR = 2020SMALL_HOLDING_USD = 500_000.0# Appendix C offshore financial centers (investor side to avoid double counting).OFC_ISO3 = {"BMU", "CYM", "GGY", "IRL", "IMN", "JEY", "LUX", "AN", "ANT"}COUNTRY_ORDER = [    "CAN", "USA", "AUT", "BEL", "DNK", "FIN", "FRA", "DEU", "ISR", "ITA", "NLD", "NOR", "PRT", "ESP", "SWE", "CHE", "GBR",    "AUS", "HKG", "JPN", "NZL", "SGP", "BRA", "CHN", "COL", "CZE", "GRC", "HUN", "IND", "MYS", "MEX", "PHL", "POL", "RUS", "ZAF", "KOR", "THA",]EQUITY_SOURCE = {    "CAN": "OECD", "USA": "OECD", "AUT": "OECD", "BEL": "OECD", "DNK": "OECD", "FIN": "OECD", "FRA": "OECD", "DEU": "OECD", "ISR": "OECD", "ITA": "OECD",    "NLD": "OECD", "NOR": "OECD", "PRT": "OECD", "ESP": "OECD", "SWE": "OECD", "CHE": "OECD", "GBR": "OECD", "AUS": "WB", "HKG": "WB", "JPN": "OECD",    "NZL": "WB", "SGP": "WB", "BRA": "OECD", "CHN": "WB", "COL": "OECD", "CZE": "OECD", "GRC": "OECD", "HUN": "OECD", "IND": "WB", "MYS": "WB",    "MEX": "OECD", "PHL": "WB", "POL": "OECD", "RUS": "WB", "ZAF": "WB", "KOR": "OECD", "THA": "WB",}DEBT_SOURCE_BASE = {    "CAN": "OECD", "USA": "OECD", "AUT": "OECD", "BEL": "OECD", "DNK": "OECD", "FIN": "OECD", "FRA": "OECD", "DEU": "OECD", "ITA": "OECD", "NLD": "OECD",    "NOR": "OECD", "PRT": "OECD", "ESP": "OECD", "SWE": "OECD", "CHE": "OECD", "GBR": "OECD", "AUS": "BIS", "HKG": "BIS", "JPN": "OECD", "NZL": "BIS",    "SGP": "BIS", "CHN": "BIS", "COL": "OECD", "CZE": "OECD", "GRC": "OECD", "HUN": "OECD", "IND": "BIS", "MYS": "BIS", "MEX": "OECD", "PHL": "BIS",    "POL": "OECD", "RUS": "BIS", "ZAF": "BIS", "KOR": "OECD", "THA": "BIS",}DEBT_SOURCE_SWITCHES = {    "BRA": {"from_year": 2009, "before": "BIS", "after": "OECD"},    "ISR": {"from_year": 2010, "before": "BIS", "after": "OECD"},}SAMPLE_START_YEAR = {    "CAN": 2003, "USA": 2003, "AUT": 2003, "BEL": 2003, "DNK": 2003, "FIN": 2003, "FRA": 2003, "DEU": 2003, "ISR": 2003, "ITA": 2003,    "NLD": 2003, "NOR": 2003, "PRT": 2003, "ESP": 2003, "SWE": 2003, "CHE": 2003, "GBR": 2003, "AUS": 2003, "HKG": 2003, "JPN": 2003,    "NZL": 2003, "SGP": 2003, "BRA": 2003, "CHN": 2015, "COL": 2007, "CZE": 2003, "GRC": 2003, "HUN": 2003, "IND": 2004, "MYS": 2005,    "MEX": 2003, "PHL": 2009, "POL": 2003, "RUS": 2004, "ZAF": 2003, "KOR": 2003, "THA": 2003,}ISO3_TO_ISO2 = {    "AUS": "AU", "AUT": "AT", "BEL": "BE", "BRA": "BR", "CAN": "CA", "CHE": "CH", "CHN": "CN", "COL": "CO",    "CZE": "CZ", "DEU": "DE", "DNK": "DK", "ESP": "ES", "FIN": "FI", "FRA": "FR", "GBR": "GB", "GRC": "GR",    "HKG": "HK", "HUN": "HU", "IND": "IN", "ISR": "IL", "ITA": "IT", "JPN": "JP", "KOR": "KR", "MEX": "MX",    "MYS": "MY", "NLD": "NL", "NOR": "NO", "NZL": "NZ", "PHL": "PH", "POL": "PL", "PRT": "PT", "RUS": "RU",    "SGP": "SG", "SWE": "SE", "THA": "TH", "USA": "US", "ZAF": "ZA",}BIS_OBS_TO_MILLION = 1000.0ASSET_TO_PIP = {"st": "ST_DEBT", "lt": "LT_DEBT", "eq": "EQUITY"}ASSET_CODE_RESTATEMENT = {"ST_DEBT": "B", "LT_DEBT": "B", "EQUITY": "E"}project_root = Path.cwd().resolve().parentdata_dir = project_root / "_data"pip_dir = project_root / "data"SHC_A7_PATH = project_root / "manual added data" / "shc_app07_2020.csv"SHC_A8_PATH = project_root / "manual added data" / "shc_app08_2020.csv"BIS_IDS_PATH = data_dir / "bis_ids_foreign_currency_q.parquet"BIS_IDS_ALL_PATH = data_dir / "bis_ids_all_currency_q.parquet"BIS_TOTAL_PATH = data_dir / "bis_total_debt_all_currency_q.parquet"oecd = pd.read_parquet(data_dir / "oecd_t720.parquet")fx = pd.read_parquet(data_dir / "oecd_implied_fx_xdc_per_usd.parquet")bis = pd.read_parquet(data_dir / "bis_debt_securities_cleaned.parquet")wb = pd.read_parquet(data_dir / "wb_data360_wdi_selected.parquet")oecd_base = oecd[    (oecd["frequency"].astype(str) == "A")    & (oecd["sector"].astype(str) == "S1")    & (oecd["consolidation"].astype(str) == "N")    & (oecd["accounting_entry"].astype(str) == "L")    & (oecd["transaction"].astype(str) == "LE")    & (oecd["counterpart_area"].astype(str) == "W")    & (oecd["counterpart_sector"].astype(str) == "S1")    & (oecd["table_identifier"].astype(str) == "T0720")].copy()wb_eq = wb[wb["metric"].astype(str) == "market_cap_listed_domestic_companies_current_usd"].copy()msci_path = data_dir / "data_msci_datastream.parquet"if msci_path.exists():    msci = pd.read_parquet(msci_path)    msci_eq = msci[msci["metric"].astype(str) == "market equity outstanding"].copy()    iso2_to_iso3 = {str(v): str(k) for k, v in ISO3_TO_ISO2.items()}    msci_eq["ref_area"] = msci_eq["entity_id"].astype(str).map(iso2_to_iso3)    msci_eq = msci_eq.dropna(subset=["ref_area", "obs_date", "value"]).copy()    msci_eq["obs_date"] = pd.to_datetime(msci_eq["obs_date"], errors="coerce")    msci_eq = msci_eq.dropna(subset=["obs_date"]).copy()    msci_eq["time_period"] = msci_eq["obs_date"].dt.year.astype("Int64")    msci_eq["value"] = pd.to_numeric(msci_eq["value"], errors="coerce")    msci_eq = msci_eq.dropna(subset=["value", "time_period"]).copy()    msci_eq = (        msci_eq.sort_values(["ref_area", "time_period", "obs_date"])        .groupby(["ref_area", "time_period"], as_index=False)        .tail(1)    )else:    msci_eq = pd.DataFrame(columns=["ref_area", "time_period", "value"])_msci_splice_ratio_cache = {}restatement_path = project_root / "manual added data" / "Restatement_Matrices.dta"restatement = pd.read_stata(restatement_path)[[    "Methodology", "Year", "Asset_Class_Code", "Destination", "Destination_Restated", "Value"]].copy()restatement["Year"] = pd.to_numeric(restatement["Year"], errors="coerce").astype("Int64")restatement["Value"] = pd.to_numeric(restatement["Value"], errors="coerce")_restatement_cache_method = {}_RESTATEMENT_ENHANCED = {"NOR", "USA"}_RESTATEMENT_FUND = {"AUT", "BEL", "FIN", "FRA", "DEU", "ITA", "NLD", "PRT", "ESP", "GRC", "AUS", "CAN", "DNK", "SWE", "CHE", "GBR", "NOR"}def _restatement_method_for_issuer(issuer: str) -> str:    iso = str(issuer)    if iso in _RESTATEMENT_ENHANCED:        return "Enhanced Fund Holdings"    if iso in _RESTATEMENT_FUND:        return "Fund Holdings"    return "Issuance"def _restatement_matrix_for_year_asset(year: int, asset_code: str, methodology: str) -> pd.DataFrame:    key = (int(year), str(asset_code), str(methodology))    if key in _restatement_cache_method:        return _restatement_cache_method[key]    x = restatement[        (restatement["Asset_Class_Code"] == asset_code)        & (restatement["Methodology"].astype(str) == str(methodology))    ].dropna(subset=["Year", "Value"]).copy()    if x.empty and str(methodology) != "Issuance":        x = restatement[            (restatement["Asset_Class_Code"] == asset_code)            & (restatement["Methodology"].astype(str) == "Issuance")        ].dropna(subset=["Year", "Value"]).copy()    if x.empty:        out = pd.DataFrame(columns=["Destination", "Destination_Restated", "Value"])        _restatement_cache_method[key] = out        return out    x["dist"] = (x["Year"].astype(int) - int(year)).abs()    y = int(x.sort_values(["dist", "Year"]).iloc[0]["Year"])    yx = x[x["Year"].astype(int) == y][["Destination", "Destination_Restated", "Value"]].copy()    _restatement_cache_method[key] = yx    return yxdef _apply_restatement_by_issuer(agg_df: pd.DataFrame) -> pd.DataFrame:    if agg_df.empty:        return agg_df.copy()    parts = []    for (tp, ac), g in agg_df.groupby(["TIME_PERIOD", "asset_class"], dropna=False):        year = int(str(tp))        asset_code = ASSET_CODE_RESTATEMENT.get(str(ac), "B")        gg = g.copy()        gg["methodology"] = "Issuance"        for method, gm in gg.groupby("methodology", dropna=False):            mat = _restatement_matrix_for_year_asset(year, asset_code, str(method))            if mat.empty:                gm2 = gm.copy()                gm2["issuer_restated"] = gm2["issuer"].astype(str)                gm2["value_restated_usd"] = pd.to_numeric(gm2["value_usd"], errors="coerce")                out = gm2.groupby(["issuer_restated", "TIME_PERIOD", "asset_class"], as_index=False)["value_restated_usd"].sum(min_count=1)                out = out.rename(columns={"issuer_restated": "issuer", "value_restated_usd": "value_usd"})                parts.append(out)                continue            m = gm.merge(mat, left_on="issuer", right_on="Destination", how="left")            m["Destination_Restated"] = m["Destination_Restated"].fillna(m["issuer"])            m["Value"] = m["Value"].fillna(1.0)            m["value_restated_usd"] = pd.to_numeric(m["value_usd"], errors="coerce") * pd.to_numeric(m["Value"], errors="coerce")            out = m.groupby(["Destination_Restated", "TIME_PERIOD", "asset_class"], as_index=False)["value_restated_usd"].sum(min_count=1)            out = out.rename(columns={"Destination_Restated": "issuer", "value_restated_usd": "value_usd"})            parts.append(out)    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["issuer", "TIME_PERIOD", "asset_class", "value_usd"])def _prepare_pip_bilateral_clean(file_name: str, value_col: str = "value_usd", issuer_col: str = "issuer") -> pd.DataFrame:    x = pd.read_parquet(        pip_dir / file_name,        columns=[issuer_col, "COUNTRY", "TIME_PERIOD", "asset_class", value_col],    )    x = x.rename(columns={issuer_col: "issuer", value_col: "value_usd"})    x = x[x["TIME_PERIOD"].astype(str) == str(TARGET_YEAR)].copy()    if x.empty:        return pd.DataFrame(columns=["issuer", "TIME_PERIOD", "asset_class", "value_usd"])    # Appendix C: drop OFC investors to avoid double counting.    x = x[~x["COUNTRY"].astype(str).isin(OFC_ISO3)].copy()    if x.empty:        return pd.DataFrame(columns=["issuer", "TIME_PERIOD", "asset_class", "value_usd"])    # Keep only sample issuers/year needed by the final table.    x = x[x["issuer"].astype(str).isin(COUNTRY_ORDER)].copy()    x = x[~x["COUNTRY"].astype(str).isin(OFC_ISO3)].copy()    if x.empty:        return pd.DataFrame(columns=["issuer", "TIME_PERIOD", "asset_class", "value_usd"])    x = x[x["COUNTRY"].astype(str) != x["issuer"].astype(str)].copy()    x["value_usd"] = pd.to_numeric(x["value_usd"], errors="coerce")    _pos = x["value_usd"] > 0    x.loc[_pos, "value_usd"] = np.ceil(x.loc[_pos, "value_usd"] / 1000.0) * 1000.0    # Appendix C: round up positive restated holdings to reporting minimum ($1,000).    _pos = x["value_usd"] > 0    x.loc[_pos, "value_usd"] = np.ceil(x.loc[_pos, "value_usd"] / 1000.0) * 1000.0    # Appendix C: keep confidential/small (<0.5m USD) holdings as missing.    x.loc[(x["value_usd"] > 0) & (x["value_usd"] < float(SMALL_HOLDING_USD)), "value_usd"] = pd.NA    agg = x.groupby(["issuer", "TIME_PERIOD", "asset_class"], as_index=False)["value_usd"].sum(min_count=1)    return aggpip_foreign_agg = _prepare_pip_bilateral_clean("pip_bilateral_positions.parquet")pip_foreign_local_agg = _prepare_pip_bilateral_clean("pip_local_foreign_allocated.parquet", value_col="value_local", issuer_col="COUNTERPART_COUNTRY")pip_reserve_agg = _prepare_pip_bilateral_clean("pip_bilateral_positions_reserve.parquet")USE_LOCAL_FOREIGN_DEBT = Truedef _pip_sum_million(agg_df: pd.DataFrame, issuer: str, year: int, asset_class: str) -> Optional[float]:    x = agg_df[        (agg_df["issuer"].astype(str) == issuer)        & (agg_df["TIME_PERIOD"].astype(str) == str(year))        & (agg_df["asset_class"].astype(str) == asset_class)    ]    if x.empty:        return None    v = pd.to_numeric(x["value_usd"], errors="coerce").sum(min_count=1)    return None if pd.isna(v) else float(v) / 1e6def _domestic_share(total_million: Optional[float], foreign_million: Optional[float]) -> Optional[float]:    if total_million is None or foreign_million is None:        return None    t = float(total_million)    f = float(foreign_million)    if pd.isna(t) or pd.isna(f) or t <= 0:        return None    # Keep raw ratio by instruction formula; do not clip negatives or values above one.    return (t - f) / tdef _share_in_reserves(reserve_million: Optional[float], foreign_total_million: Optional[float]) -> Optional[float]:    if reserve_million is None or foreign_total_million is None:        return None    r = float(reserve_million)    f = float(foreign_total_million)    if pd.isna(r) or pd.isna(f) or f <= 0:        return None    out = r / f    return max(0.0, min(1.0, out))# SHC naming differs from ISO3 labels for a few issuers.ISO3_TO_SHC_NAME = {    "CAN": "Canada", "USA": "United States", "AUT": "Austria", "BEL": "Belgium", "DNK": "Denmark",    "FIN": "Finland", "FRA": "France", "DEU": "Germany", "ISR": "Israel", "ITA": "Italy",    "NLD": "Netherlands", "NOR": "Norway", "PRT": "Portugal", "ESP": "Spain", "SWE": "Sweden",    "CHE": "Switzerland", "GBR": "United Kingdom", "AUS": "Australia", "HKG": "Hong Kong",    "JPN": "Japan", "NZL": "New Zealand", "SGP": "Singapore", "BRA": "Brazil",    "CHN": "China, mainland (1)", "COL": "Colombia", "CZE": "Czech Republic", "GRC": "Greece",    "HUN": "Hungary", "IND": "India", "MYS": "Malaysia", "MEX": "Mexico",    "PHL": "Philippines", "POL": "Poland", "RUS": "Russia", "ZAF": "South Africa",    "KOR": "Korea, South", "THA": "Thailand",}def _read_shc_country_total_million(path: Path, country_name: str) -> Optional[float]:    if not path.exists():        return None    rows = []    with open(path, "r", encoding="utf-8-sig", newline="") as f:        reader = csv.reader(f)        for r in reader:            rows.append(r)    if not rows:        return None    header_idx = None    for i, r in enumerate(rows):        if len(r) > 0 and str(r[0]).strip() == "Country or region of issuer":            header_idx = i            break    if header_idx is None:        return None    vals = []    for r in rows[header_idx + 1 :]:        if len(r) < 2:            continue        c = str(r[0]).strip()        if c == "":            continue        if c.casefold() != str(country_name).casefold():            continue        v = pd.to_numeric(r[1], errors="coerce")        if pd.notna(v):            vals.append(float(v))    if not vals:        return None    return float(sum(vals))def _usa_bilateral_st_lt_million() -> pd.DataFrame:    p = pip_dir / "pip_bilateral_positions.parquet"    if not p.exists():        return pd.DataFrame(columns=["issuer", "asset_class", "usa_mn"])    x = pd.read_parquet(p, columns=["COUNTRY", "issuer", "TIME_PERIOD", "asset_class", "value_usd"])    x = x[        (x["COUNTRY"].astype(str) == "USA")        & (x["TIME_PERIOD"].astype(str) == str(TARGET_YEAR))        & (x["issuer"].astype(str).isin(COUNTRY_ORDER))        & (x["asset_class"].astype(str).isin(["ST_DEBT", "LT_DEBT"]))    ].copy()    if x.empty:        return pd.DataFrame(columns=["issuer", "asset_class", "usa_mn"])    x["value_usd"] = pd.to_numeric(x["value_usd"], errors="coerce")    x.loc[(x["value_usd"] > 0) & (x["value_usd"] < float(SMALL_HOLDING_USD)), "value_usd"] = pd.NA    g = x.groupby(["issuer", "asset_class"], as_index=False)["value_usd"].sum(min_count=1)    g["usa_mn"] = pd.to_numeric(g["value_usd"], errors="coerce") / 1e6    return g[["issuer", "asset_class", "usa_mn"]]# --- OECD helpers ---def _oecd_unit_for_country_year(country_iso3: str, year: int) -> Optional[str]:    x = oecd_base[(oecd_base["reference_area"] == country_iso3) & (oecd_base["time_period"].astype(str) == str(year))]    if x.empty:        return None    if (x["unit_measure"] == "XDC").any():        return "XDC"    if (x["unit_measure"] == "USD").any():        return "USD"    return Nonedef _to_usd_million_oecd(country_iso3: str, year: int, raw_million: float, unit_measure: str) -> Optional[float]:    if unit_measure == "USD":        return float(raw_million)    if unit_measure == "XDC":        row = fx[(fx["reference_area"] == country_iso3) & (fx["time_period"].astype(str) == str(year))]        if row.empty:            return None        f = float(row["fx_xdc_per_usd"].iloc[0])        if f <= 0:            return None        return float(raw_million) / f    return Nonedef _nearest_year_ratio_for_debt_split_oecd(country_iso3: str, target_year: int, debt_ccy: str) -> Optional[float]:    x_all = oecd_base[(oecd_base["reference_area"] == country_iso3) & (oecd_base["currency_denom"] == debt_ccy)]    if x_all.empty:        return None    years = sorted(x_all["time_period"].dropna().astype(str).unique().tolist())    candidates = []    for y in years:        x = x_all[x_all["time_period"].astype(str) == y]        st = float(x[(x["financial_instrument"] == "F3") & (x["original_maturity"] == "S")]["value"].sum())        lt = float(x[(x["financial_instrument"] == "F3") & (x["original_maturity"] == "L")]["value"].sum())        if st > 0 and lt > 0:            candidates.append((int(y), st / (st + lt)))    if not candidates:        return None    return min(candidates, key=lambda t: abs(t[0] - int(target_year)))[1]def _nearest_year_ratio_for_equity_split_oecd(country_iso3: str, target_year: int) -> Optional[float]:    x_all = oecd_base[oecd_base["reference_area"] == country_iso3]    if x_all.empty:        return None    years = sorted(x_all["time_period"].dropna().astype(str).unique().tolist())    candidates = []    for y in years:        x = x_all[x_all["time_period"].astype(str) == y]        f51 = float(x[(x["financial_instrument"] == "F51") & (x["original_maturity"].isin(["_Z", "T"]))]["value"].sum())        f519 = float(x[(x["financial_instrument"] == "F519") & (x["original_maturity"].isin(["_Z", "T"]))]["value"].sum())        ce = max(0.0, f51 - f519)        if ce <= 0 and f51 > 0:            ce = f51        f5 = float(x[(x["financial_instrument"] == "F5") & (x["original_maturity"].isin(["_Z", "T"]))]["value"].sum())        if ce > 0 and f5 > 0:            candidates.append((int(y), ce / f5))    if not candidates:        return None    return min(candidates, key=lambda t: abs(t[0] - int(target_year)))[1]def _oecd_debt_st_lt_usd_million(country_iso3: str, year: int) -> tuple[Optional[float], Optional[float]]:    u = _oecd_unit_for_country_year(country_iso3, year)    if u is None:        return None, None    x = oecd_base[        (oecd_base["reference_area"] == country_iso3)        & (oecd_base["time_period"].astype(str) == str(year))        & (oecd_base["unit_measure"] == u)    ]    if x.empty:        return None, None    debt_ccy = next((ccy for ccy in ["_T", "XDC", "USD"] if (x["currency_denom"] == ccy).any()), None)    if debt_ccy is None:        return None, None    st_raw = float(x[(x["financial_instrument"] == "F3") & (x["original_maturity"] == "S") & (x["currency_denom"] == debt_ccy)]["value"].sum())    lt_raw = float(x[(x["financial_instrument"] == "F3") & (x["original_maturity"] == "L") & (x["currency_denom"] == debt_ccy)]["value"].sum())    tot_raw = float(x[(x["financial_instrument"] == "F3") & (x["original_maturity"].isin(["T", "_Z"])) & (x["currency_denom"] == debt_ccy)]["value"].sum())    if (st_raw == 0.0 or lt_raw == 0.0) and tot_raw > 0.0:        r = _nearest_year_ratio_for_debt_split_oecd(country_iso3, int(year), debt_ccy)        if r is not None:            st_raw = tot_raw * r            lt_raw = tot_raw * (1.0 - r)    return _to_usd_million_oecd(country_iso3, year, st_raw, u), _to_usd_million_oecd(country_iso3, year, lt_raw, u)def _oecd_equity_usd_million(country_iso3: str, year: int) -> Optional[float]:    u = _oecd_unit_for_country_year(country_iso3, year)    if u is None:        return None    x = oecd_base[        (oecd_base["reference_area"] == country_iso3)        & (oecd_base["time_period"].astype(str) == str(year))        & (oecd_base["unit_measure"] == u)    ]    if x.empty:        return None    f51 = float(x[(x["financial_instrument"] == "F51") & (x["original_maturity"].isin(["_Z", "T"]))]["value"].sum())    f519 = float(x[(x["financial_instrument"] == "F519") & (x["original_maturity"].isin(["_Z", "T"]))]["value"].sum())    ce_raw = max(0.0, f51 - f519)    if ce_raw <= 0.0 and f51 > 0.0:        ce_raw = f51    f5_raw = float(x[(x["financial_instrument"] == "F5") & (x["original_maturity"].isin(["_Z", "T"]))]["value"].sum())    if ce_raw <= 0.0 and f5_raw > 0.0:        r = _nearest_year_ratio_for_equity_split_oecd(country_iso3, int(year))        ce_raw = f5_raw * r if r is not None else f5_raw    return _to_usd_million_oecd(country_iso3, year, ce_raw, u)# --- BIS helper ---def _bis_base(country_iso2: str) -> pd.DataFrame:    x = bis[        (bis["REF_AREA"].astype(str) == country_iso2)        & (bis["COUNTERPART_SECTOR"].astype(str) == "S1")        & (bis["CONSOLIDATION"].astype(str) == "N")        & (bis["ACCOUNTING_ENTRY"].astype(str) == "L")        & (bis["STO"].astype(str) == "LE")        & (bis["INSTR_ASSET"].astype(str) == "F3")        & (bis["TRANSFORMATION"].astype(str) == "N")        & (bis["UNIT_MEASURE"].astype(str) == "USD")        & (bis["PRICES"].astype(str) == "V")        & (bis["EXPENDITURE"].astype(str) == "_Z")        & (bis["REF_SECTOR"].astype(str).isin(["S11", "S12", "S13"]))    ].copy()    x["OBS_VALUE"] = pd.to_numeric(x["OBS_VALUE"], errors="coerce")    return xdef _bis_pick_currency_slice(x: pd.DataFrame) -> pd.DataFrame:    if x.empty:        return x    present = set(x["CURRENCY_DENOM"].astype(str).unique().tolist())    # Prefer total first, then local-currency coded totals in this dataset.    for c in ["_T", "XDC", "X1", "USD"]:        if c in present:            return x[x["CURRENCY_DENOM"].astype(str) == c].copy()    return xdef _bis_pick_valuation_slice(x: pd.DataFrame) -> pd.DataFrame:    if x.empty:        return x    vals = set(x["VALUATION"].astype(str).unique().tolist())    for v in ["M", "N", "F"]:        if v in vals:            return x[x["VALUATION"].astype(str) == v].copy()    return xdef _bis_nearest_st_ratio(country_iso2: str, target_year: int, currency_denom: Optional[str] = None) -> Optional[float]:    q = _bis_base(country_iso2)    q = q[(q["FREQ"].astype(str) == "Q") & (q["TIME_PERIOD"].astype(str).str.endswith("-Q4"))].copy()    if currency_denom is not None:        q = q[q["CURRENCY_DENOM"].astype(str) == str(currency_denom)].copy()    else:        q = _bis_pick_currency_slice(q)    q = _bis_pick_valuation_slice(q)    if q.empty:        return None    out = []    for tp, g in q.groupby("TIME_PERIOD"):        st = float(g[g["MATURITY"].astype(str) == "S"]["OBS_VALUE"].sum())        lt = float(g[g["MATURITY"].astype(str) == "L"]["OBS_VALUE"].sum())        if st > 0 and lt > 0:            out.append((int(str(tp)[:4]), st / (st + lt)))    if not out:        return None    return min(out, key=lambda t: abs(t[0] - int(target_year)))[1]def _bis_st_share_from_available_issuers(country_iso2: str, period: str, freq: str, currency_denom: Optional[str] = None) -> Optional[float]:    x = _bis_base(country_iso2)    x = x[(x["FREQ"].astype(str) == str(freq)) & (x["TIME_PERIOD"].astype(str) == str(period))].copy()    if currency_denom is not None:        x = x[x["CURRENCY_DENOM"].astype(str) == str(currency_denom)].copy()    else:        x = _bis_pick_currency_slice(x)    x = _bis_pick_valuation_slice(x)    if x.empty:        return None    st_sum = 0.0    lt_sum = 0.0    for _, g in x.groupby("REF_SECTOR", dropna=False):        st_i = float(g[g["MATURITY"].astype(str) == "S"]["OBS_VALUE"].sum())        lt_i = float(g[g["MATURITY"].astype(str) == "L"]["OBS_VALUE"].sum())        if st_i > 0.0 and lt_i > 0.0:            st_sum += st_i            lt_sum += lt_i    den = st_sum + lt_sum    if den <= 0.0:        return None    return st_sum / dendef _bis_global_st_ratio(period: str, freq: str, currency_denom: Optional[str] = None) -> Optional[float]:    x = bis[        (bis["COUNTERPART_SECTOR"].astype(str) == "S1")        & (bis["CONSOLIDATION"].astype(str) == "N")        & (bis["ACCOUNTING_ENTRY"].astype(str) == "L")        & (bis["STO"].astype(str) == "LE")        & (bis["INSTR_ASSET"].astype(str) == "F3")        & (bis["TRANSFORMATION"].astype(str) == "N")        & (bis["UNIT_MEASURE"].astype(str) == "USD")        & (bis["PRICES"].astype(str) == "V")        & (bis["EXPENDITURE"].astype(str) == "_Z")        & (bis["REF_SECTOR"].astype(str).isin(["S11", "S12", "S13"]))        & (bis["FREQ"].astype(str) == str(freq))        & (bis["TIME_PERIOD"].astype(str) == str(period))    ].copy()    if x.empty:        return None    if currency_denom is not None:        x = x[x["CURRENCY_DENOM"].astype(str) == str(currency_denom)].copy()    x = _bis_pick_valuation_slice(x)    if x.empty:        return None    x["OBS_VALUE"] = pd.to_numeric(x["OBS_VALUE"], errors="coerce")    st = float(x[x["MATURITY"].astype(str) == "S"]["OBS_VALUE"].sum())    lt = float(x[x["MATURITY"].astype(str) == "L"]["OBS_VALUE"].sum())    den = st + lt    if den <= 0.0:        return None    return st / dendef _bis_domestic_st_lt_usd_million(country_iso2: str, year: int) -> tuple[Optional[float], Optional[float]]:    q = _bis_base(country_iso2)    periods = [("Q", f"{int(year)}-Q4"), ("A", str(int(year)))]    for freq, period in periods:        x = q[(q["FREQ"].astype(str) == str(freq)) & (q["TIME_PERIOD"].astype(str) == str(period))].copy()        if x.empty:            continue        x = _bis_pick_currency_slice(x)        x = _bis_pick_valuation_slice(x)        if x.empty:            continue        ccy = str(x["CURRENCY_DENOM"].astype(str).iloc[0]) if "CURRENCY_DENOM" in x.columns else None        r_issuer = _bis_st_share_from_available_issuers(country_iso2, str(period), str(freq), currency_denom=ccy)        r_nearest = _bis_nearest_st_ratio(country_iso2, int(year), currency_denom=ccy)        r_global = _bis_global_st_ratio(str(period), str(freq), currency_denom=ccy)        st_obs = float(x[x["MATURITY"].astype(str) == "S"]["OBS_VALUE"].sum())        lt_obs = float(x[x["MATURITY"].astype(str) == "L"]["OBS_VALUE"].sum())        total_obs = float(x[x["MATURITY"].astype(str).isin(["T", "TT", "LS"])]["OBS_VALUE"].sum())        st_sum = 0.0        lt_sum = 0.0        for _, g in x.groupby("REF_SECTOR", dropna=False):            st_i = float(g[g["MATURITY"].astype(str) == "S"]["OBS_VALUE"].sum())            lt_i = float(g[g["MATURITY"].astype(str) == "L"]["OBS_VALUE"].sum())            tot_i = float(g[g["MATURITY"].astype(str).isin(["T", "TT", "LS"])]["OBS_VALUE"].sum())            if st_i > 0.0 and lt_i > 0.0:                st_sum += st_i                lt_sum += lt_i                continue            if tot_i > 0.0:                rr = r_issuer if r_issuer is not None else (r_nearest if r_nearest is not None else r_global)                if rr is not None:                    st_sum += tot_i * float(rr)                    lt_sum += tot_i * (1.0 - float(rr))                    continue            st_sum += st_i            lt_sum += lt_i        if (st_sum <= 0.0 or lt_sum <= 0.0) and total_obs > 0.0:            rr = r_nearest if r_nearest is not None else (r_issuer if r_issuer is not None else r_global)            if rr is not None:                st_sum = total_obs * float(rr)                lt_sum = total_obs * (1.0 - float(rr))        if st_sum > 0.0 or lt_sum > 0.0:            return float(st_sum) * float(BIS_OBS_TO_MILLION), float(lt_sum) * float(BIS_OBS_TO_MILLION)    return None, Nonedef _bis_has_raw_st_lt_split(country_iso2: str, year: int) -> bool:    q = _bis_base(country_iso2)    periods = [("Q", f"{int(year)}-Q4"), ("A", str(int(year)))]    for freq, period in periods:        x = q[(q["FREQ"].astype(str) == str(freq)) & (q["TIME_PERIOD"].astype(str) == str(period))].copy()        if x.empty:            continue        x = _bis_pick_currency_slice(x)        x = _bis_pick_valuation_slice(x)        if x.empty:            continue        st_obs = float(x[x["MATURITY"].astype(str) == "S"]["OBS_VALUE"].sum())        lt_obs = float(x[x["MATURITY"].astype(str) == "L"]["OBS_VALUE"].sum())        if st_obs > 0.0 and lt_obs > 0.0:            return True    return Falsedef _bis_ids_foreign_st_lt_usd_million(country_iso2: str, year: int) -> tuple[float, float]:    if not BIS_IDS_PATH.exists():        return 0.0, 0.0    ids = pd.read_parquet(BIS_IDS_PATH).copy()    for c in [        "MEASURE", "MARKET", "ISSUE_TYPE", "ISSUE_CUR", "ISSUE_RE_MAT", "ISSUE_RATE", "ISSUE_RISK", "ISSUE_COL",        "ISSUER_BUS_IMM", "ISSUER_BUS_ULT", "ISSUE_CUR_GROUP", "ISSUE_OR_MAT", "TIME_PERIOD", "ISSUER_RES",    ]:        if c in ids.columns:            ids[c] = ids[c].astype(str)    ids = ids[        (ids["ISSUER_RES"] == str(country_iso2))        & (ids["MEASURE"] == "I")        & (ids["MARKET"] == "C")        & (ids["ISSUE_TYPE"] == "A")        & (ids["ISSUE_CUR_GROUP"] == "F")        & (ids["TIME_PERIOD"] == f"{year}-Q4")    ].copy()    if ids.empty:        return 0.0, 0.0    if "ISSUE_CUR" in ids.columns and (ids["ISSUE_CUR"].astype(str) == "TO1").any():        ids = ids[ids["ISSUE_CUR"].astype(str) == "TO1"].copy()    if "ISSUE_RE_MAT" in ids.columns and (ids["ISSUE_RE_MAT"].astype(str) == "A").any():        ids = ids[ids["ISSUE_RE_MAT"].astype(str) == "A"].copy()    for _c in ["ISSUE_RATE", "ISSUE_RISK", "ISSUE_COL"]:        if _c in ids.columns and (ids[_c].astype(str) == "A").any():            ids = ids[ids[_c].astype(str) == "A"].copy()    # Avoid double counting: prefer aggregate issuer-business rows when available.    if "ISSUER_BUS_IMM" in ids.columns:        ids_agg = ids[ids["ISSUER_BUS_IMM"].astype(str) == "1"].copy()        if not ids_agg.empty:            ids = ids_agg    if "million_usd" in ids.columns:        ids["million_usd"] = pd.to_numeric(ids["million_usd"], errors="coerce")    else:        ids["million_usd"] = (            pd.to_numeric(ids.get("OBS_VALUE"), errors="coerce")            * (10 ** pd.to_numeric(ids.get("UNIT_MULT"), errors="coerce").fillna(0))        ) / 1e6    st = float(ids[ids["ISSUE_OR_MAT"] == "C"]["million_usd"].sum())    lt = float(ids[ids["ISSUE_OR_MAT"] == "K"]["million_usd"].sum())    return st, ltdef _bis_ids_all_st_lt_usd_million(country_iso2: str, year: int) -> tuple[float, float]:    # Preferred source: all-currency IDS pull (A/D/F groups).    p = BIS_IDS_ALL_PATH if 'BIS_IDS_ALL_PATH' in globals() else None    if p is not None and Path(p).exists():        ids = pd.read_parquet(p).copy()        for c in ["ISSUER_RES", "ISSUE_OR_MAT", "TIME_PERIOD", "ISSUE_CUR_GROUP", "ISSUER_BUS_IMM"]:            if c in ids.columns:                ids[c] = ids[c].astype(str)        ids = ids[            (ids.get("ISSUER_RES", "").astype(str) == str(country_iso2))            & (ids.get("TIME_PERIOD", "").astype(str) == f"{year}-Q4")            & (ids.get("ISSUE_OR_MAT", "").astype(str).isin(["A", "C", "K"]))            & (ids.get("ISSUE_CUR_GROUP", "").astype(str) == "A")            & (ids.get("MEASURE", "").astype(str) == "I")            & (ids.get("MARKET", "").astype(str) == "C")            & (ids.get("ISSUE_TYPE", "").astype(str) == "A")        ].copy()        if not ids.empty:            if "ISSUE_CUR" in ids.columns and (ids["ISSUE_CUR"].astype(str) == "TO1").any():                ids = ids[ids["ISSUE_CUR"].astype(str) == "TO1"].copy()            if "ISSUE_RE_MAT" in ids.columns and (ids["ISSUE_RE_MAT"].astype(str) == "A").any():                ids = ids[ids["ISSUE_RE_MAT"].astype(str) == "A"].copy()            for _c in ["ISSUE_RATE", "ISSUE_RISK", "ISSUE_COL"]:                if _c in ids.columns and (ids[_c].astype(str) == "A").any():                    ids = ids[ids[_c].astype(str) == "A"].copy()            # Avoid double counting: prefer aggregate issuer-business rows when available.            if "ISSUER_BUS_IMM" in ids.columns:                ids_agg = ids[ids["ISSUER_BUS_IMM"].astype(str) == "1"].copy()                if not ids_agg.empty:                    ids = ids_agg            if "million_usd" in ids.columns:                ids["million_usd"] = pd.to_numeric(ids["million_usd"], errors="coerce")            else:                ids["million_usd"] = (                    pd.to_numeric(ids.get("OBS_VALUE"), errors="coerce")                    * (10 ** pd.to_numeric(ids.get("UNIT_MULT"), errors="coerce").fillna(0))                ) / 1e6            st = float(ids[ids["ISSUE_OR_MAT"] == "C"]["million_usd"].sum())            lt = float(ids[ids["ISSUE_OR_MAT"] == "K"]["million_usd"].sum())            tot = float(ids[ids["ISSUE_OR_MAT"] == "A"]["million_usd"].sum())            # Appendix C fallback: if C/K split is missing but total is present,            # allocate by nearest available short-term share.            if tot > 0.0 and (st <= 0.0 or lt <= 0.0):                r = _bis_total_nearest_st_ratio(country_iso2, int(year))                if r is None:                    r = _bis_global_st_ratio(f"{int(year)}-Q4", "Q", currency_denom=None)                if r is not None:                    st = tot * float(r)                    lt = tot * (1.0 - float(r))                else:                    if st > 0.0 and lt <= 0.0:                        lt = max(0.0, tot - st)                    elif lt > 0.0 and st <= 0.0:                        st = max(0.0, tot - lt)            return st, lt    # Fallback: foreign-currency IDS only (less preferred but better than missing).    return _bis_ids_foreign_st_lt_usd_million(country_iso2, year)def _bis_total_nearest_st_ratio(country_iso2: str, target_year: int) -> Optional[float]:    if not BIS_IDS_ALL_PATH.exists():        return None    ids = pd.read_parquet(BIS_IDS_ALL_PATH).copy()    for c in ["ISSUER_RES", "ISSUE_OR_MAT", "TIME_PERIOD", "ISSUE_CUR_GROUP", "MEASURE", "MARKET", "ISSUE_TYPE", "ISSUE_CUR", "ISSUE_RE_MAT", "ISSUE_RATE", "ISSUE_RISK", "ISSUE_COL", "ISSUER_BUS_IMM"]:        if c in ids.columns:            ids[c] = ids[c].astype(str)    ids = ids[        (ids.get("ISSUER_RES", "").astype(str) == str(country_iso2))        & (ids.get("ISSUE_CUR_GROUP", "").astype(str) == "A")        & (ids.get("ISSUE_OR_MAT", "").astype(str).isin(["C", "K"]))        & (ids.get("MEASURE", "").astype(str) == "I")        & (ids.get("MARKET", "").astype(str) == "C")        & (ids.get("ISSUE_TYPE", "").astype(str) == "A")    ].copy()    if ids.empty:        return None    if "ISSUE_CUR" in ids.columns and (ids["ISSUE_CUR"].astype(str) == "TO1").any():        ids = ids[ids["ISSUE_CUR"].astype(str) == "TO1"].copy()    if "ISSUE_RE_MAT" in ids.columns and (ids["ISSUE_RE_MAT"].astype(str) == "A").any():        ids = ids[ids["ISSUE_RE_MAT"].astype(str) == "A"].copy()    for _c in ["ISSUE_RATE", "ISSUE_RISK", "ISSUE_COL"]:        if _c in ids.columns and (ids[_c].astype(str) == "A").any():            ids = ids[ids[_c].astype(str) == "A"].copy()    if "ISSUER_BUS_IMM" in ids.columns:        ids_agg = ids[ids["ISSUER_BUS_IMM"].astype(str) == "1"].copy()        if not ids_agg.empty:            ids = ids_agg    if "million_usd" in ids.columns:        ids["million_usd"] = pd.to_numeric(ids["million_usd"], errors="coerce")    else:        ids["million_usd"] = (            pd.to_numeric(ids.get("OBS_VALUE"), errors="coerce")            * (10 ** pd.to_numeric(ids.get("UNIT_MULT"), errors="coerce").fillna(0))        ) / 1e6    cand = []    for tp, g in ids.groupby("TIME_PERIOD", dropna=False):        y = int(str(tp)[:4])        st = float(g[g["ISSUE_OR_MAT"].astype(str) == "C"]["million_usd"].sum())        lt = float(g[g["ISSUE_OR_MAT"].astype(str) == "K"]["million_usd"].sum())        if st > 0.0 and lt > 0.0:            cand.append((abs(y - int(target_year)), y, st / (st + lt)))    if not cand:        return None    cand.sort(key=lambda t: (t[0], t[1]))    return float(cand[0][2])def _bis_total_st_lt_usd_million(country_iso2: str, year: int) -> tuple[float, float]:    if not BIS_TOTAL_PATH.exists():        return 0.0, 0.0    tot = pd.read_parquet(BIS_TOTAL_PATH).copy()    for c in ["ISSUER_RES", "TIME_PERIOD", "COUNTERPART_AREA", "REF_SECTOR", "ACCOUNTING_ENTRY", "STO", "INSTR_ASSET", "MATURITY", "CURRENCY_DENOM", "CUST_BREAKDOWN"]:        if c in tot.columns:            tot[c] = tot[c].astype(str)    tot = tot[(tot.get("ISSUER_RES", "").astype(str) == str(country_iso2))].copy()    tot_filters = {        "ACCOUNTING_ENTRY": "L",        "STO": "LE",        "INSTR_ASSET": "F3",        "MATURITY": "T",        "CURRENCY_DENOM": "_T",        "CUST_BREAKDOWN": "_T",    }    for c, v in tot_filters.items():        if c in tot.columns:            tot = tot[tot[c].astype(str) == v].copy()    if tot.empty:        return 0.0, 0.0    if "million_usd" in tot.columns:        tot["million_usd"] = pd.to_numeric(tot["million_usd"], errors="coerce")    else:        tot["million_usd"] = (            pd.to_numeric(tot.get("OBS_VALUE"), errors="coerce")            * (10 ** pd.to_numeric(tot.get("UNIT_MULT"), errors="coerce").fillna(0))        ) / 1e6    _tp = tot.get("TIME_PERIOD", pd.Series(index=tot.index, dtype=object)).astype(str)    _target = tot[_tp.isin([f"{year}-Q4", str(year)])].copy()    if _target.empty:        _yr = pd.to_numeric(_tp.str.slice(0, 4), errors="coerce")        _ok = _yr.notna()        if _ok.any():            _nearest = int(_yr[_ok].astype(int).drop_duplicates().sort_values(key=lambda s: (s - int(year)).abs()).iloc[0])            _target = tot[_yr == _nearest].copy()    if _target.empty:        return 0.0, 0.0    tot = _target    if "COUNTERPART_AREA" in tot.columns:        _tot_xw = tot[tot["COUNTERPART_AREA"].astype(str) == "XW"].copy()        if not _tot_xw.empty:            tot = _tot_xw        else:            _cp_sum = (                tot.groupby("COUNTERPART_AREA", as_index=False)["million_usd"]                .sum(min_count=1)                .sort_values("million_usd", ascending=False)            )            if not _cp_sum.empty and pd.notna(_cp_sum.iloc[0]["million_usd"]):                _best_cp = str(_cp_sum.iloc[0]["COUNTERPART_AREA"])                tot = tot[tot["COUNTERPART_AREA"].astype(str) == _best_cp].copy()    if "REF_SECTOR" in tot.columns:        _rs = tot["REF_SECTOR"].astype(str)        _s_total = float(tot.loc[_rs.isin({"S1", "S1M", "S1ZS"}), "million_usd"].sum())        _comp_total = float(tot.loc[_rs.isin({"S11", "S12", "S13"}), "million_usd"].sum())        if _s_total > 0.0 and _comp_total > 0.0:            total_obs = max(_s_total, _comp_total)        elif _s_total > 0.0:            total_obs = _s_total        elif _comp_total > 0.0:            total_obs = _comp_total        else:            total_obs = float(tot["million_usd"].sum())    else:        total_obs = float(tot["million_usd"].sum())    if total_obs <= 0.0:        return 0.0, 0.0    r = _bis_total_nearest_st_ratio(country_iso2, int(year))    if r is None:        ids_st_mn, ids_lt_mn = _bis_ids_all_st_lt_usd_million(country_iso2, int(year))        ids_tot = float(ids_st_mn) + float(ids_lt_mn)        if ids_st_mn > 0.0 and ids_lt_mn > 0.0 and ids_tot > 0.0:            r = float(ids_st_mn) / ids_tot    if r is None:        r = _bis_global_st_ratio(f"{year}-Q4", "Q", currency_denom=None)    if r is None:        return 0.0, 0.0    st = total_obs * float(r)    lt = total_obs * (1.0 - float(r))    return st, ltdef _bis_debt_st_lt_usd_million(country_iso3: str, year: int) -> tuple[Optional[float], Optional[float]]:    iso2 = ISO3_TO_ISO2.get(country_iso3)    if iso2 is None:        return None, None    # Appendix C: total debt = domestic debt securities + international debt securities.    ids_st_mn, ids_lt_mn = _bis_ids_all_st_lt_usd_million(iso2, int(year))    dom_st_mn, dom_lt_mn = _bis_domestic_st_lt_usd_million(iso2, int(year))    tot_st_mn, tot_lt_mn = _bis_total_st_lt_usd_million(iso2, int(year))    tot_all = float(tot_st_mn) + float(tot_lt_mn)    ids_all = float(ids_st_mn) + float(ids_lt_mn)    dom_all_obs = float(dom_st_mn or 0.0) + float(dom_lt_mn or 0.0)    dom_raw_split_ok = _bis_has_raw_st_lt_split(iso2, int(year))    dom_equals_total = (tot_all > 0.0) and (abs(dom_all_obs - tot_all) <= 1e-6 * max(1.0, abs(tot_all)))    # Precision-gated missing/quality detection for BIS domestic leg:    # 1) true missing/non-positive domestic observations    dom_missing_basic = (dom_st_mn is None or dom_lt_mn is None) or (dom_all_obs <= 0.0)    # 2) split-quality issue only matters if ST/LT is still unusable after domestic construction    dom_split_still_missing = (dom_all_obs > 0.0) and ((float(dom_st_mn or 0.0) <= 0.0) or (float(dom_lt_mn or 0.0) <= 0.0))    dom_split_unreliable = (not dom_raw_split_ok) and dom_split_still_missing    # 3) consistency checks against total-minus-IDS identity, with coverage gate    dom_recon_all_raw = max(0.0, tot_all - ids_all) if tot_all > 0.0 else 0.0    _dom_tol_identity = max(0.20 * max(1.0, abs(tot_all)), 250.0)    _ids_total_conflict = (tot_all > 0.0) and (ids_all > 1.15 * tot_all)    dom_inconsistent_with_identity = (        (tot_all > 0.0)        and (not _ids_total_conflict)        and _ids_material        and (abs(dom_all_obs - dom_recon_all_raw) > max(_dom_tol_identity, 0.35 * max(1.0, abs(dom_recon_all_raw))))    )    # 4) domestic equals total is suspicious only when IDS is materially positive    _ids_material = (tot_all > 0.0) and (ids_all > max(0.05 * tot_all, 100.0))    dom_equals_total_suspicious = dom_equals_total and _ids_material and (not _ids_total_conflict)    # 5) implausible scale: domestic meaningfully exceeds total debt    _dom_tol_overshoot = max(0.10 * max(1.0, abs(tot_all)), 250.0)    dom_overshoots_total = (tot_all > 0.0) and (dom_all_obs > (tot_all + _dom_tol_overshoot))    use_reconstructed_domestic = (        dom_missing_basic        or dom_split_unreliable        or dom_overshoots_total        or dom_equals_total_suspicious    )    # Appendix C rule: impute domestic when missing or when strict quality checks fail.    if use_reconstructed_domestic and tot_all > 0.0:        dom_recon_all = dom_recon_all_raw        if dom_recon_all <= 0.0:            dom_st_mn = 0.0            dom_lt_mn = 0.0        else:            rr = None            if ids_all > 0.0 and float(ids_st_mn) > 0.0 and float(ids_lt_mn) > 0.0:                rr = float(ids_st_mn) / ids_all            if rr is None and dom_all_obs > 0.0 and (dom_st_mn is not None) and (dom_lt_mn is not None) and float(dom_st_mn) > 0.0 and float(dom_lt_mn) > 0.0:                rr = float(dom_st_mn) / dom_all_obs            if rr is None:                rr = _bis_total_nearest_st_ratio(iso2, int(year))            if rr is None:                rr = _bis_global_st_ratio(f"{int(year)}-Q4", "Q", currency_denom=None)            if rr is not None:                dom_st_mn = dom_recon_all * float(rr)                dom_lt_mn = dom_recon_all * (1.0 - float(rr))            else:                dom_st_mn = dom_recon_all                dom_lt_mn = 0.0    st_mn = float(ids_st_mn) + float(dom_st_mn or 0.0)    lt_mn = float(ids_lt_mn) + float(dom_lt_mn or 0.0)    if st_mn <= 0.0 and lt_mn <= 0.0:        return None, None    return float(st_mn), float(lt_mn)def _wb_equity_usd_million(country_iso3: str, year: int) -> Optional[float]:    x = wb_eq[(wb_eq["ref_area"] == country_iso3) & (wb_eq["time_period"].astype(str) == str(year))].copy()    if x.empty:        return None    v = pd.to_numeric(x["value"], errors="coerce").sum(min_count=1)    return None if pd.isna(v) else float(v) / 1e6def _msci_equity_raw(country_iso3: str, year: int) -> Optional[float]:    x = msci_eq[(msci_eq["ref_area"] == country_iso3) & (msci_eq["time_period"].astype(str) == str(year))]    if x.empty:        return None    v = pd.to_numeric(x["value"], errors="coerce").sum(min_count=1)    return None if pd.isna(v) else float(v)def _msci_splice_ratio(country_iso3: str, year: int, anchor_name: str, anchor_getter) -> Optional[float]:    key = (country_iso3, int(year), str(anchor_name))    if key in _msci_splice_ratio_cache:        return _msci_splice_ratio_cache[key]    overlaps = []    years = sorted({int(y) for y in pd.to_numeric(msci_eq[msci_eq["ref_area"] == country_iso3]["time_period"], errors="coerce").dropna().astype(int).tolist()})    for yy in years:        m = _msci_equity_raw(country_iso3, yy)        a = anchor_getter(int(yy))        if m is None or a is None:            continue        if float(m) <= 0.0 or float(a) <= 0.0:            continue        overlaps.append((abs(int(yy) - int(year)), int(yy), float(a) / float(m)))    if not overlaps:        _msci_splice_ratio_cache[key] = None        return None    overlaps.sort(key=lambda t: (t[0], t[1]))    ratio = float(overlaps[0][2])    _msci_splice_ratio_cache[key] = ratio    return ratiodef _msci_equity_spliced_usd_million(country_iso3: str, year: int, anchor_name: str, anchor_getter) -> Optional[float]:    m = _msci_equity_raw(country_iso3, year)    if m is None or float(m) <= 0.0:        return None    ratio = _msci_splice_ratio(country_iso3, year, anchor_name, anchor_getter)    if ratio is None or float(ratio) <= 0.0:        return None    return float(m) * float(ratio)# Countries with instruction-level debt source switches over time.SWITCH_COUNTRIES = set(DEBT_SOURCE_SWITCHES.keys())# Countries with delayed sample entry in Table C1.LATE_ENTRY_COUNTRIES = {c for c, y in SAMPLE_START_YEAR.items() if int(y) > 2003}def _debt_source(country_iso3: str, year: int) -> str:    sw = DEBT_SOURCE_SWITCHES.get(country_iso3)    if sw is not None:        return sw["after"] if int(year) >= int(sw["from_year"]) else sw["before"]    return DEBT_SOURCE_BASE[country_iso3]# --- Step A totals (residency) ---# Performance mode: only compute TARGET_YEAR (final table year).base_rows = []for country_iso3 in COUNTRY_ORDER:    year = int(TARGET_YEAR)    debt_src = _debt_source(country_iso3, year)    eq_src = EQUITY_SOURCE[country_iso3]    if debt_src == "OECD":        st_mn, lt_mn = _oecd_debt_st_lt_usd_million(country_iso3, year)    else:        st_mn, lt_mn = _bis_debt_st_lt_usd_million(country_iso3, year)    if eq_src == "OECD":        eq_mn = _oecd_equity_usd_million(country_iso3, year)        if eq_mn is None:            eq_mn = _msci_equity_spliced_usd_million(                country_iso3,                year,                "OECD",                lambda yy: _oecd_equity_usd_million(country_iso3, int(yy)),            )    else:        eq_mn = _wb_equity_usd_million(country_iso3, year)        if eq_mn is None:            eq_mn = _msci_equity_spliced_usd_million(                country_iso3,                year,                "WB",                lambda yy: _wb_equity_usd_million(country_iso3, int(yy)),            )    base_rows.append(        {            "country": country_iso3,            "year": int(year),            "debt_source": debt_src,            "equity_source": eq_src,            "sample_start_year": int(SAMPLE_START_YEAR[country_iso3]),            "st_mn_residency": st_mn,            "lt_mn_residency": lt_mn,            "eq_mn_residency": eq_mn,        }    )base_panel = pd.DataFrame(base_rows)# Appendix C: restate totals from issuer residency to nationality.totals_long = base_panel.melt(    id_vars=["country", "year", "debt_source", "equity_source", "sample_start_year"],    value_vars=["st_mn_residency", "lt_mn_residency", "eq_mn_residency"],    var_name="metric",    value_name="value_mn",)metric_to_asset = {"st_mn_residency": "ST_DEBT", "lt_mn_residency": "LT_DEBT", "eq_mn_residency": "EQUITY"}totals_long["asset_class"] = totals_long["metric"].map(metric_to_asset)totals_long = totals_long.rename(columns={"country": "issuer", "year": "TIME_PERIOD"})totals_long["value_usd"] = pd.to_numeric(totals_long["value_mn"], errors="coerce") * 1e6restated_totals = _apply_restatement_by_issuer(totals_long[["issuer", "TIME_PERIOD", "asset_class", "value_usd"]])restated_totals["value_mn"] = pd.to_numeric(restated_totals["value_usd"], errors="coerce") / 1e6restated_totals_w = restated_totals.pivot_table(    index=["issuer", "TIME_PERIOD"],    columns="asset_class",    values="value_mn",    aggfunc="sum",).reset_index()restated_totals_w = restated_totals_w.rename(columns={"issuer": "country", "TIME_PERIOD": "year", "ST_DEBT": "st_mn", "LT_DEBT": "lt_mn", "EQUITY": "eq_mn"})restated_totals_w["year"] = pd.to_numeric(restated_totals_w["year"], errors="coerce").astype("Int64")panel = base_panel.merge(restated_totals_w[["country", "year", "st_mn", "lt_mn", "eq_mn"]], on=["country", "year"], how="left")# Fill missing post-restatement with residency values when a country does not appear in matrix outputs.panel["st_mn"] = panel["st_mn"].where(pd.notna(panel["st_mn"]), panel["st_mn_residency"])panel["lt_mn"] = panel["lt_mn"].where(pd.notna(panel["lt_mn"]), panel["lt_mn_residency"])panel["eq_mn"] = panel["eq_mn"].where(pd.notna(panel["eq_mn"]), panel["eq_mn_residency"])# Local-currency debt diagnostic track (do not overwrite total debt outstanding).st_local_vals = []lt_local_vals = []bis_missing_base_flags = []oecd_negative_subtraction = []for _, rr in panel.iterrows():    iso2 = ISO3_TO_ISO2.get(str(rr["country"]))    st_fc_mn = 0.0    lt_fc_mn = 0.0    if iso2 is not None and pd.notna(rr["year"]):        st_fc_mn, lt_fc_mn = _bis_ids_foreign_st_lt_usd_million(iso2, int(rr["year"]))    st_raw = pd.to_numeric(rr["st_mn"], errors="coerce")    lt_raw = pd.to_numeric(rr["lt_mn"], errors="coerce")    debt_src = str(rr.get("debt_source", ""))    has_bis_base = pd.notna(rr.get("st_mn_residency")) or pd.notna(rr.get("lt_mn_residency"))    missing_bis_base = debt_src == "BIS" and not has_bis_base    bis_missing_base_flags.append(bool(missing_bis_base))    # If BIS domestic base is missing, use instruction fallback:    # domestic = total debt - international debt; split by IDS maturity shares.    if missing_bis_base:        total_raw = 0.0        if pd.notna(st_raw):            total_raw += float(st_raw)        if pd.notna(lt_raw):            total_raw += float(lt_raw)        ids_st_mn, ids_lt_mn = _bis_ids_all_st_lt_usd_million(str(iso2), int(rr["year"])) if iso2 is not None else (0.0, 0.0)        ids_tot = float(ids_st_mn) + float(ids_lt_mn)        if iso2 is not None and pd.notna(rr["year"]):            tot_st_mn, tot_lt_mn = _bis_total_st_lt_usd_million(str(iso2), int(rr["year"]))            tot_all = float(tot_st_mn) + float(tot_lt_mn)            dom_all = max(0.0, tot_all - ids_tot)            if dom_all > 0.0 and ids_tot > 0.0:                st_share_ids = float(ids_st_mn) / ids_tot                dom_st = dom_all * st_share_ids                dom_lt = dom_all * (1.0 - st_share_ids)                st_local_vals.append(dom_st + float(ids_st_mn))                lt_local_vals.append(dom_lt + float(ids_lt_mn))                continue            if tot_all > 0.0 and (tot_st_mn > 0.0 or tot_lt_mn > 0.0):                st_local_vals.append(float(tot_st_mn))                lt_local_vals.append(float(tot_lt_mn))                continue        if total_raw > 0.0 and ids_tot > 0.0:            st_share_ids = float(ids_st_mn) / ids_tot            st_local_vals.append(total_raw * st_share_ids)            lt_local_vals.append(total_raw * (1.0 - st_share_ids))        else:            st_local_vals.append(None if pd.isna(st_raw) else float(st_raw))            lt_local_vals.append(None if pd.isna(lt_raw) else float(lt_raw))        continue    st_local = None if pd.isna(st_raw) else float(st_raw)    lt_local = None if pd.isna(lt_raw) else float(lt_raw)    # Local-currency debt alignment: subtract BIS foreign-currency IDS on residency totals first,    # then map to nationality using the same-year restatement ratio.    if debt_src == "OECD":        st_resid = pd.to_numeric(rr.get("st_mn_residency"), errors="coerce")        lt_resid = pd.to_numeric(rr.get("lt_mn_residency"), errors="coerce")        st_fc_adj = float(st_fc_mn)        lt_fc_adj = float(lt_fc_mn)        if pd.notna(st_resid) and float(st_fc_adj) > float(st_resid):            oecd_negative_subtraction.append((str(rr.get("country", "")), "ST(strict-invalid)", float(st_resid), float(st_fc_adj)))            st_fc_adj = float("nan")        if pd.notna(lt_resid) and float(lt_fc_adj) > float(lt_resid):            oecd_negative_subtraction.append((str(rr.get("country", "")), "LT(strict-invalid)", float(lt_resid), float(lt_fc_adj)))            lt_fc_adj = float("nan")        if pd.notna(st_resid):            if pd.isna(st_fc_adj):                st_local = float(st_raw) if pd.notna(st_raw) else float(st_resid)            else:                st_resid_adj = float(st_resid) - float(st_fc_adj)                st_ratio = 1.0                if pd.notna(st_raw) and float(st_resid) > 0.0:                    st_ratio = float(st_raw) / float(st_resid)                st_local = max(0.0, st_resid_adj) * st_ratio        if pd.notna(lt_resid):            if pd.isna(lt_fc_adj):                lt_local = float(lt_raw) if pd.notna(lt_raw) else float(lt_resid)            else:                lt_resid_adj = float(lt_resid) - float(lt_fc_adj)                lt_ratio = 1.0                if pd.notna(lt_raw) and float(lt_resid) > 0.0:                    lt_ratio = float(lt_raw) / float(lt_resid)                lt_local = max(0.0, lt_resid_adj) * lt_ratio    # Keep BIS totals on the same post-restatement chain; do not add IDS again here.    st_local_vals.append(st_local)    lt_local_vals.append(lt_local)panel["st_local_mn"] = st_local_valspanel["lt_local_mn"] = lt_local_valspanel["bis_missing_base"] = bis_missing_base_flagsif oecd_negative_subtraction:    print("WARNING: OECD-BIS subtraction negative on residency totals for:")    for cc, mat, resid_v, bis_v in oecd_negative_subtraction:        print(f"  {cc} {mat}: residency={resid_v:.3f} mn, bis_ids_foreign={bis_v:.3f} mn")# Appendix C: foreign/reserve holdings are also restated from issuer residency to nationality.def _attach_pip_sum(panel_df: pd.DataFrame, agg_df: pd.DataFrame, asset: str, out_col: str) -> pd.DataFrame:    x = agg_df[agg_df["asset_class"].astype(str) == asset].copy()    x["value_mn"] = pd.to_numeric(x["value_usd"], errors="coerce") / 1e6    x = x.rename(columns={"issuer": "country", "TIME_PERIOD": "year"})[["country", "year", "value_mn"]]    x["year"] = pd.to_numeric(x["year"], errors="coerce").astype("Int64")    return panel_df.merge(x.rename(columns={"value_mn": out_col}), on=["country", "year"], how="left")debt_foreign_agg = pip_foreign_local_agg if USE_LOCAL_FOREIGN_DEBT else pip_foreign_aggpanel = _attach_pip_sum(panel, debt_foreign_agg, "ST_DEBT", "foreign_st_mn")panel = _attach_pip_sum(panel, debt_foreign_agg, "LT_DEBT", "foreign_lt_mn")panel = _attach_pip_sum(panel, pip_foreign_agg, "EQUITY", "foreign_eq_mn")panel = _attach_pip_sum(panel, pip_reserve_agg, "ST_DEBT", "reserve_st_mn")panel = _attach_pip_sum(panel, pip_reserve_agg, "LT_DEBT", "reserve_lt_mn")panel = _attach_pip_sum(panel, pip_reserve_agg, "EQUITY", "reserve_eq_mn")if "bis_missing_base" in panel.columns:    _m = panel["bis_missing_base"] == True    panel.loc[_m, ["foreign_st_mn", "foreign_lt_mn", "reserve_st_mn", "reserve_lt_mn"]] = None# Data-instruction alignment: SHC replacement is total-holdings based.# Skip it when debt foreign inputs are local-currency allocated to avoid mixing definitions.if not USE_LOCAL_FOREIGN_DEBT:    usa_stlt = _usa_bilateral_st_lt_million()    usa_lookup = {(str(r["issuer"]), str(r["asset_class"])): float(r["usa_mn"]) for _, r in usa_stlt.iterrows() if pd.notna(r["usa_mn"])}    for asset, shc_path, out_col in [        ("ST_DEBT", SHC_A7_PATH, "foreign_st_mn"),        ("LT_DEBT", SHC_A8_PATH, "foreign_lt_mn"),    ]:        for i, row in panel.iterrows():            iso3 = str(row["country"])            shc_name = ISO3_TO_SHC_NAME.get(iso3, iso3)            shc_val = _read_shc_country_total_million(shc_path, shc_name)            if shc_val is None:                continue            foreign_before = row[out_col]            if pd.isna(foreign_before):                continue            usa_before = usa_lookup.get((iso3, asset), 0.0)            panel.at[i, out_col] = max(0.0, float(foreign_before) - float(usa_before) + float(shc_val))panel["debt_billion_usd"] = (pd.to_numeric(panel["st_mn"], errors="coerce") + pd.to_numeric(panel["lt_mn"], errors="coerce")) / 1000.0panel["st_debt_billion_usd"] = pd.to_numeric(panel["st_local_mn"], errors="coerce") / 1000.0panel["lt_debt_billion_usd"] = pd.to_numeric(panel["lt_local_mn"], errors="coerce") / 1000.0panel["equity_billion_usd"] = pd.to_numeric(panel["eq_mn"], errors="coerce") / 1000.0# Keep missing foreign-local debt as missing; do not coerce to zero._st_foreign_for_share = pd.to_numeric(panel["foreign_st_mn"], errors="coerce")_lt_foreign_for_share = pd.to_numeric(panel["foreign_lt_mn"], errors="coerce")panel["domestic_share_st_debt"] = [    _domestic_share(t, f) for t, f in zip(panel["st_local_mn"], _st_foreign_for_share)]panel["domestic_share_lt_debt"] = [    _domestic_share(t, f) for t, f in zip(panel["lt_local_mn"], _lt_foreign_for_share)]panel["domestic_share_equity"] = [    _domestic_share(t, f) for t, f in zip(panel["eq_mn"], panel["foreign_eq_mn"])]_reserve_foreign_st = pd.to_numeric(panel["foreign_st_total_mn"], errors="coerce") if (USE_LOCAL_FOREIGN_DEBT and "foreign_st_total_mn" in panel.columns) else pd.to_numeric(panel["foreign_st_mn"], errors="coerce")_reserve_foreign_lt = pd.to_numeric(panel["foreign_lt_total_mn"], errors="coerce") if (USE_LOCAL_FOREIGN_DEBT and "foreign_lt_total_mn" in panel.columns) else pd.to_numeric(panel["foreign_lt_mn"], errors="coerce")panel["share_in_reserves_st_debt"] = [    _share_in_reserves(r, f) for r, f in zip(panel["reserve_st_mn"], _reserve_foreign_st)]panel["share_in_reserves_lt_debt"] = [    _share_in_reserves(r, f) for r, f in zip(panel["reserve_lt_mn"], _reserve_foreign_lt)]panel["share_in_reserves_equity"] = [    _share_in_reserves(r, f) for r, f in zip(panel["reserve_eq_mn"], panel["foreign_eq_mn"])]full_country_panel = panel.sort_values(["country", "year"]).reset_index(drop=True)asset_rows = []for _, r in full_country_panel.iterrows():    asset_rows.extend(        [            {                "country": r["country"],                "year": int(r["year"]),                "asset_class": "Short-term debt",                "value_billion_usd": r["st_debt_billion_usd"],                "domestic_share": r["domestic_share_st_debt"],                "share_in_reserves": r["share_in_reserves_st_debt"],            },            {                "country": r["country"],                "year": int(r["year"]),                "asset_class": "Long-term debt",                "value_billion_usd": r["lt_debt_billion_usd"],                "domestic_share": r["domestic_share_lt_debt"],                "share_in_reserves": r["share_in_reserves_lt_debt"],            },            {                "country": r["country"],                "year": int(r["year"]),                "asset_class": "Equity",                "value_billion_usd": r["equity_billion_usd"],                "domestic_share": r["domestic_share_equity"],                "share_in_reserves": r["share_in_reserves_equity"],            },        ]    )full_country_panel_long = pd.DataFrame(asset_rows)panel_2020 = full_country_panel[full_country_panel["year"] == TARGET_YEAR].copy()panel_2020_long = full_country_panel_long[full_country_panel_long["year"] == TARGET_YEAR].copy()

In [101]:
# Reserve input override: prefer IMF confidentiality aggregate reserve investor (e.g., TX093)
# when available, while keeping table1 debt foreign source as local-currency debt.
reserve_agg_path = pip_dir / "pip_bilateral_positions_reserve_aggregate.parquet"
if reserve_agg_path.exists():
    pip_reserve_agg = _prepare_pip_bilateral_clean("pip_bilateral_positions_reserve_aggregate.parquet")
    print(f"Using reserve aggregate source for table1: {reserve_agg_path}")
else:
    print("Reserve aggregate file not found; using reserve-sector bilateral source.")

Using reserve aggregate source for table1: D:\Git\p09_koijen_yogo_2020\data\pip_bilateral_positions_reserve_aggregate.parquet


In [102]:
# Make downstream recomputation idempotent when rerunning notebook cells.
if "panel" in globals():
    panel = panel.drop(
        columns=[
            "foreign_st_total_mn", "foreign_lt_total_mn",
            "foreign_st_local_direct_mn", "foreign_lt_local_direct_mn",
            "st_local_share", "lt_local_share",
        ],
        errors="ignore",
    )
# Rebuild domestic-share inputs from scratch (Appendix C definitions).
# Debt domestic share = local-currency debt foreign holdings, combining SHC-adjusted totals
# with CPIS-implied local shares (uniform rule for all issuers).
def _prepare_foreign_for_share(file_name: str, value_col: str, issuer_col: str) -> pd.DataFrame:
    x = pd.read_parquet(
        pip_dir / file_name,
        columns=[issuer_col, "COUNTRY", "TIME_PERIOD", "asset_class", value_col],
    ).rename(columns={issuer_col: "issuer", value_col: "value_usd"})

    x = x[x["TIME_PERIOD"].astype(str) == str(TARGET_YEAR)].copy()
    x = x[x["issuer"].astype(str).isin(COUNTRY_ORDER)].copy()
    # Appendix C: drop OFC investors to avoid double counting.
    x = x[~x["COUNTRY"].astype(str).isin(OFC_ISO3)].copy()
    x = x[x["COUNTRY"].astype(str) != x["issuer"].astype(str)].copy()
    x["value_usd"] = pd.to_numeric(x["value_usd"], errors="coerce")

    # Appendix C: round up positive holdings to reporting minimum ($1,000).
    _pos = x["value_usd"] > 0
    x.loc[_pos, "value_usd"] = np.ceil(x.loc[_pos, "value_usd"] / 1000.0) * 1000.0

    # Appendix C: confidential and small holdings are missing, not zero, before aggregation.
    x.loc[(x["value_usd"] > 0) & (x["value_usd"] < float(SMALL_HOLDING_USD)), "value_usd"] = pd.NA

    # Aggregate using observed bilateral data; min_count=1 avoids manufacturing zero when all are missing.
    g = (
        x.groupby(["issuer", "TIME_PERIOD", "asset_class"], as_index=False)["value_usd"]
        .sum(min_count=1)
    )
    return g


def _attach_agg_million(panel_df: pd.DataFrame, agg_df: pd.DataFrame, asset: str, out_col: str) -> pd.DataFrame:
    y = agg_df[agg_df["asset_class"].astype(str) == asset].copy()
    y["value_mn"] = pd.to_numeric(y["value_usd"], errors="coerce") / 1e6
    y = y.rename(columns={"issuer": "country", "TIME_PERIOD": "year"})[["country", "year", "value_mn"]]
    y["year"] = pd.to_numeric(y["year"], errors="coerce").astype("Int64")
    return panel_df.merge(y.rename(columns={"value_mn": out_col}), on=["country", "year"], how="left")


foreign_debt_local_agg = _prepare_foreign_for_share(
    "pip_local_foreign_allocated.parquet",
    value_col="value_local",
    issuer_col="COUNTERPART_COUNTRY",
)
foreign_total_agg = _prepare_foreign_for_share(
    "pip_bilateral_positions.parquet",
    value_col="value_usd",
    issuer_col="issuer",
)
reserve_agg = _prepare_foreign_for_share(
    "pip_bilateral_positions_reserve.parquet",
    value_col="value_usd",
    issuer_col="issuer",
)

# Replace foreign/reserve inputs with rebuilt aggregates.
panel = panel.drop(
    columns=[
        "foreign_st_mn", "foreign_lt_mn", "foreign_eq_mn",
        "foreign_st_total_mn", "foreign_lt_total_mn",
        "foreign_st_local_direct_mn", "foreign_lt_local_direct_mn",
        "st_local_share", "lt_local_share",
        "reserve_st_mn", "reserve_lt_mn", "reserve_eq_mn",
        "domestic_share_st_debt", "domestic_share_lt_debt", "domestic_share_equity",
        "share_in_reserves_st_debt", "share_in_reserves_lt_debt", "share_in_reserves_equity",
    ],
    errors="ignore",
)

# Step 1: attach debt foreign totals (pre-local split).
panel = _attach_agg_million(panel, foreign_total_agg, "ST_DEBT", "foreign_st_total_mn")
panel = _attach_agg_million(panel, foreign_total_agg, "LT_DEBT", "foreign_lt_total_mn")
# Step 2: attach direct local totals from allocation output.
panel = _attach_agg_million(panel, foreign_debt_local_agg, "ST_DEBT", "foreign_st_local_direct_mn")
panel = _attach_agg_million(panel, foreign_debt_local_agg, "LT_DEBT", "foreign_lt_local_direct_mn")
panel = _attach_agg_million(panel, foreign_total_agg, "EQUITY", "foreign_eq_mn")
panel = _attach_agg_million(panel, reserve_agg, "ST_DEBT", "reserve_st_mn")
panel = _attach_agg_million(panel, reserve_agg, "LT_DEBT", "reserve_lt_mn")
panel = _attach_agg_million(panel, reserve_agg, "EQUITY", "reserve_eq_mn")

# Step 3: SHC replacement on debt foreign totals (USA component adjustment).
usa_stlt = _usa_bilateral_st_lt_million()
usa_lookup = {(str(r["issuer"]), str(r["asset_class"])): float(r["usa_mn"]) for _, r in usa_stlt.iterrows() if pd.notna(r["usa_mn"])}
for asset, shc_path, out_col in [
    ("ST_DEBT", SHC_A7_PATH, "foreign_st_total_mn"),
    ("LT_DEBT", SHC_A8_PATH, "foreign_lt_total_mn"),
]:
    for i, row in panel.iterrows():
        iso3 = str(row["country"])
        shc_name = ISO3_TO_SHC_NAME.get(iso3, iso3)
        shc_val = _read_shc_country_total_million(shc_path, shc_name)
        if shc_val is None:
            continue
        foreign_before = row[out_col]
        if pd.isna(foreign_before):
            continue
        usa_before = usa_lookup.get((iso3, asset), 0.0)
        panel.at[i, out_col] = max(0.0, float(foreign_before) - float(usa_before) + float(shc_val))

# In local-foreign debt mode, keep foreign debt totals from CPIS aggregate pulls.
# SHC replacement is total-holdings based and can be definition-incompatible here.
if USE_LOCAL_FOREIGN_DEBT:
    panel = panel.drop(columns=["foreign_st_total_mn", "foreign_lt_total_mn"], errors="ignore")
    panel = _attach_agg_million(panel, foreign_total_agg, "ST_DEBT", "foreign_st_total_mn")
    panel = _attach_agg_million(panel, foreign_total_agg, "LT_DEBT", "foreign_lt_total_mn")

# Step 4: estimate local shares from CPIS allocation and apply to SHC-adjusted totals.
debt_local_w = foreign_debt_local_agg[foreign_debt_local_agg["asset_class"].isin(["ST_DEBT", "LT_DEBT"])].copy()
debt_total_w = foreign_total_agg[foreign_total_agg["asset_class"].isin(["ST_DEBT", "LT_DEBT"])].copy()
debt_local_w["local_mn"] = pd.to_numeric(debt_local_w["value_usd"], errors="coerce") / 1e6
debt_total_w["total_mn"] = pd.to_numeric(debt_total_w["value_usd"], errors="coerce") / 1e6
shares = debt_local_w[["issuer", "TIME_PERIOD", "asset_class", "local_mn"]].merge(
    debt_total_w[["issuer", "TIME_PERIOD", "asset_class", "total_mn"]],
    on=["issuer", "TIME_PERIOD", "asset_class"],
    how="outer",
)
shares["local_share"] = pd.to_numeric(shares["local_mn"], errors="coerce") / pd.to_numeric(shares["total_mn"], errors="coerce")
shares = shares.rename(columns={"issuer": "country", "TIME_PERIOD": "year"})[["country", "year", "asset_class", "local_share"]]
shares["year"] = pd.to_numeric(shares["year"], errors="coerce").astype("Int64")
st_share = shares[shares["asset_class"] == "ST_DEBT"][["country", "year", "local_share"]].rename(columns={"local_share": "st_local_share"})
lt_share = shares[shares["asset_class"] == "LT_DEBT"][["country", "year", "local_share"]].rename(columns={"local_share": "lt_local_share"})
panel = panel.merge(st_share, on=["country", "year"], how="left")
panel = panel.merge(lt_share, on=["country", "year"], how="left")
# Keep CPIS-implied local shares, with OECD fallback from issuer-side local/total ratio.
_m_debt = panel["debt_source"].astype(str) == "OECD"
for _share_col, _num_col, _den_col in [
    ("st_local_share", "st_local_mn", "st_mn_residency"),
    ("lt_local_share", "lt_local_mn", "lt_mn_residency"),
]:
    _share = pd.to_numeric(panel[_share_col], errors="coerce")
    _share = _share.where((_share >= 0.0) & (_share <= 1.0))
    _num = pd.to_numeric(panel[_num_col], errors="coerce")
    _den = pd.to_numeric(panel[_den_col], errors="coerce")
    _fallback = (_num / _den).where(_den > 0.0)
    _fallback = _fallback.where((_fallback >= 0.0) & (_fallback <= 1.0))
    panel[_share_col] = _share.where(pd.notna(_share), _fallback.where(_m_debt))
panel["foreign_st_mn"] = pd.to_numeric(panel["foreign_st_total_mn"], errors="coerce") * pd.to_numeric(panel["st_local_share"], errors="coerce")
panel["foreign_lt_mn"] = pd.to_numeric(panel["foreign_lt_total_mn"], errors="coerce") * pd.to_numeric(panel["lt_local_share"], errors="coerce")
panel["foreign_st_mn"] = panel["foreign_st_mn"].where(pd.notna(panel["foreign_st_mn"]), panel["foreign_st_local_direct_mn"])
panel["foreign_lt_mn"] = panel["foreign_lt_mn"].where(pd.notna(panel["foreign_lt_mn"]), panel["foreign_lt_local_direct_mn"])

# Keep source debt totals unchanged here; do not floor totals to foreign-side estimates.

# OECD safeguard: if local-foreign debt exceeds local debt outstanding, use issuer-side share fallback.
_st_total = pd.to_numeric(panel["st_local_mn"], errors="coerce")
_lt_total = pd.to_numeric(panel["lt_local_mn"], errors="coerce")
_st_foreign = pd.to_numeric(panel["foreign_st_mn"], errors="coerce")
_lt_foreign = pd.to_numeric(panel["foreign_lt_mn"], errors="coerce")
_st_bad = _m_debt & _st_total.notna() & _st_foreign.notna() & ((_st_foreign < 0.0) | (_st_foreign > _st_total))
_lt_bad = _m_debt & _lt_total.notna() & _lt_foreign.notna() & ((_lt_foreign < 0.0) | (_lt_foreign > _lt_total))
if _st_bad.any():
    _st_den = pd.to_numeric(panel["st_mn_residency"], errors="coerce")
    _st_ratio = (_st_total / _st_den).where(_st_den > 0.0)
    _st_ratio = _st_ratio.where((_st_ratio >= 0.0) & (_st_ratio <= 1.0))
    panel.loc[_st_bad, "foreign_st_mn"] = pd.to_numeric(panel.loc[_st_bad, "foreign_st_total_mn"], errors="coerce") * _st_ratio.loc[_st_bad]
if _lt_bad.any():
    _lt_den = pd.to_numeric(panel["lt_mn_residency"], errors="coerce")
    _lt_ratio = (_lt_total / _lt_den).where(_lt_den > 0.0)
    _lt_ratio = _lt_ratio.where((_lt_ratio >= 0.0) & (_lt_ratio <= 1.0))
    panel.loc[_lt_bad, "foreign_lt_mn"] = pd.to_numeric(panel.loc[_lt_bad, "foreign_lt_total_mn"], errors="coerce") * _lt_ratio.loc[_lt_bad]

# Never keep impossible foreign-local debt values; leave as missing for explicit diagnostics.
for _f_col, _t_col in [("foreign_st_mn", "st_local_mn"), ("foreign_lt_mn", "lt_local_mn")]:
    _f = pd.to_numeric(panel[_f_col], errors="coerce")
    _t = pd.to_numeric(panel[_t_col], errors="coerce")
    panel.loc[_f.notna() & _t.notna() & ((_f < 0.0) | (_f > _t)), _f_col] = pd.NA

if "bis_missing_base" in panel.columns:
    _m = panel["bis_missing_base"] == True
    panel.loc[_m, ["foreign_st_mn", "foreign_lt_mn", "reserve_st_mn", "reserve_lt_mn"]] = None

# Recompute shares using rebuilt numerator inputs.
_st_total_for_share = pd.to_numeric(panel["st_local_mn"], errors="coerce")
_lt_total_for_share = pd.to_numeric(panel["lt_local_mn"], errors="coerce")
# Keep missing foreign-local debt as missing; do not coerce to zero.
_st_foreign_for_share = pd.to_numeric(panel["foreign_st_mn"], errors="coerce")
_lt_foreign_for_share = pd.to_numeric(panel["foreign_lt_mn"], errors="coerce")
panel["domestic_share_st_debt"] = [
    _domestic_share(t, f) for t, f in zip(_st_total_for_share, _st_foreign_for_share)
]
panel["domestic_share_lt_debt"] = [
    _domestic_share(t, f) for t, f in zip(_lt_total_for_share, _lt_foreign_for_share)
]
panel["domestic_share_equity"] = [
    _domestic_share(t, f) for t, f in zip(panel["eq_mn"], panel["foreign_eq_mn"])
]

_reserve_foreign_st = pd.to_numeric(panel["foreign_st_total_mn"], errors="coerce") if (USE_LOCAL_FOREIGN_DEBT and "foreign_st_total_mn" in panel.columns) else pd.to_numeric(panel["foreign_st_mn"], errors="coerce")
_reserve_foreign_lt = pd.to_numeric(panel["foreign_lt_total_mn"], errors="coerce") if (USE_LOCAL_FOREIGN_DEBT and "foreign_lt_total_mn" in panel.columns) else pd.to_numeric(panel["foreign_lt_mn"], errors="coerce")
panel["share_in_reserves_st_debt"] = [
    _share_in_reserves(r, f) for r, f in zip(panel["reserve_st_mn"], _reserve_foreign_st)
]
panel["share_in_reserves_lt_debt"] = [
    _share_in_reserves(r, f) for r, f in zip(panel["reserve_lt_mn"], _reserve_foreign_lt)
]
panel["share_in_reserves_equity"] = [
    _share_in_reserves(r, f) for r, f in zip(panel["reserve_eq_mn"], panel["foreign_eq_mn"])
]

full_country_panel = panel.sort_values(["country", "year"]).reset_index(drop=True)

asset_rows = []
for _, r in full_country_panel.iterrows():
    asset_rows.extend([
        {
            "country": r["country"],
            "year": int(r["year"]),
            "asset_class": "Short-term debt",
            "value_billion_usd": r["st_debt_billion_usd"],
            "domestic_share": r["domestic_share_st_debt"],
            "share_in_reserves": r["share_in_reserves_st_debt"],
        },
        {
            "country": r["country"],
            "year": int(r["year"]),
            "asset_class": "Long-term debt",
            "value_billion_usd": r["lt_debt_billion_usd"],
            "domestic_share": r["domestic_share_lt_debt"],
            "share_in_reserves": r["share_in_reserves_lt_debt"],
        },
        {
            "country": r["country"],
            "year": int(r["year"]),
            "asset_class": "Equity",
            "value_billion_usd": r["equity_billion_usd"],
            "domestic_share": r["domestic_share_equity"],
            "share_in_reserves": r["share_in_reserves_equity"],
        },
    ])

full_country_panel_long = pd.DataFrame(asset_rows)
panel_2020 = full_country_panel[full_country_panel["year"] == TARGET_YEAR].copy()
panel_2020_long = full_country_panel_long[full_country_panel_long["year"] == TARGET_YEAR].copy()

In [103]:
# Post-build reserve alignment: prefer IMF confidentiality aggregate reserve source
# for reserve shares while keeping debt foreign inputs in local-currency terms.
reserve_agg_path = pip_dir / "pip_bilateral_positions_reserve_aggregate.parquet"
if reserve_agg_path.exists():
    reserve_agg = _prepare_foreign_for_share(
        "pip_bilateral_positions_reserve_aggregate.parquet",
        value_col="value_usd",
        issuer_col="issuer",
    )

    # Keep debt reserves on bilateral scope; use aggregate only for equity reserve.
    panel = panel.drop(columns=["reserve_eq_mn"], errors="ignore")
    panel = _attach_agg_million(panel, reserve_agg, "EQUITY", "reserve_eq_mn")

    _reserve_foreign_st = pd.to_numeric(panel["foreign_st_total_mn"], errors="coerce") if (USE_LOCAL_FOREIGN_DEBT and "foreign_st_total_mn" in panel.columns) else pd.to_numeric(panel["foreign_st_mn"], errors="coerce")
    _reserve_foreign_lt = pd.to_numeric(panel["foreign_lt_total_mn"], errors="coerce") if (USE_LOCAL_FOREIGN_DEBT and "foreign_lt_total_mn" in panel.columns) else pd.to_numeric(panel["foreign_lt_mn"], errors="coerce")
    panel["share_in_reserves_st_debt"] = [
        _share_in_reserves(r, f) for r, f in zip(panel["reserve_st_mn"], _reserve_foreign_st)
    ]
    panel["share_in_reserves_lt_debt"] = [
        _share_in_reserves(r, f) for r, f in zip(panel["reserve_lt_mn"], _reserve_foreign_lt)
    ]
    panel["share_in_reserves_equity"] = [
        _share_in_reserves(r, f) for r, f in zip(panel["reserve_eq_mn"], panel["foreign_eq_mn"])
    ]

    full_country_panel = panel.sort_values(["country", "year"]).reset_index(drop=True)
    print(f"Applied reserve aggregate alignment (equity only) in table1 from: {reserve_agg_path}")

Applied reserve aggregate alignment (equity only) in table1 from: D:\Git\p09_koijen_yogo_2020\data\pip_bilateral_positions_reserve_aggregate.parquet


In [104]:
# Full-country harmonization pass: fill residual missing debt-foreign inputs.
# 1) If BIS-base-missing rows lost foreign debt in earlier safety guards, recover from
#    SHC-adjusted totals x local-share (already computed in the rebuild cell).
if "foreign_st_mn" in panel.columns and "foreign_st_total_mn" in panel.columns and "st_local_share" in panel.columns:
    m_st = panel["foreign_st_mn"].isna() & panel["foreign_st_total_mn"].notna() & panel["st_local_share"].notna()
    _st_fill = (
        pd.to_numeric(panel.loc[m_st, "foreign_st_total_mn"], errors="coerce")
        * pd.to_numeric(panel.loc[m_st, "st_local_share"], errors="coerce")
    )
    _st_cap = pd.to_numeric(panel.loc[m_st, "st_mn"], errors="coerce")
    panel.loc[m_st, "foreign_st_mn"] = _st_fill.where((_st_fill >= 0.0) & (_st_cap.notna()) & (_st_fill <= _st_cap))

if "foreign_lt_mn" in panel.columns and "foreign_lt_total_mn" in panel.columns and "lt_local_share" in panel.columns:
    m_lt = panel["foreign_lt_mn"].isna() & panel["foreign_lt_total_mn"].notna() & panel["lt_local_share"].notna()
    _lt_fill = (
        pd.to_numeric(panel.loc[m_lt, "foreign_lt_total_mn"], errors="coerce")
        * pd.to_numeric(panel.loc[m_lt, "lt_local_share"], errors="coerce")
    )
    _lt_cap = pd.to_numeric(panel.loc[m_lt, "lt_mn"], errors="coerce")
    panel.loc[m_lt, "foreign_lt_mn"] = _lt_fill.where((_lt_fill >= 0.0) & (_lt_cap.notna()) & (_lt_fill <= _lt_cap))

# 2) Reserve holdings: keep missing as missing (instruction-consistent confidentiality handling).
for c in ["reserve_st_mn", "reserve_lt_mn", "reserve_eq_mn"]:
    if c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

# 3) Recompute level columns after harmonization to keep outputs in sync.
panel["debt_billion_usd"] = (pd.to_numeric(panel["st_mn"], errors="coerce") + pd.to_numeric(panel["lt_mn"], errors="coerce")) / 1000.0
panel["st_debt_billion_usd"] = pd.to_numeric(panel["st_local_mn"], errors="coerce") / 1000.0
panel["lt_debt_billion_usd"] = pd.to_numeric(panel["lt_local_mn"], errors="coerce") / 1000.0
panel["equity_billion_usd"] = pd.to_numeric(panel["eq_mn"], errors="coerce") / 1000.0

# 4) Recompute shares after harmonization.
_st_total_for_share = pd.to_numeric(panel["st_local_mn"], errors="coerce")
_lt_total_for_share = pd.to_numeric(panel["lt_local_mn"], errors="coerce")
_st_foreign_for_share = pd.to_numeric(panel["foreign_st_mn"], errors="coerce")
_lt_foreign_for_share = pd.to_numeric(panel["foreign_lt_mn"], errors="coerce")
_st_foreign_for_share = _st_foreign_for_share.where((_st_foreign_for_share >= 0.0) & (_st_total_for_share.isna() | (_st_foreign_for_share <= _st_total_for_share)))
_lt_foreign_for_share = _lt_foreign_for_share.where((_lt_foreign_for_share >= 0.0) & (_lt_total_for_share.isna() | (_lt_foreign_for_share <= _lt_total_for_share)))
panel["domestic_share_st_debt"] = [_domestic_share(t, f) for t, f in zip(_st_total_for_share, _st_foreign_for_share)]
panel["domestic_share_lt_debt"] = [_domestic_share(t, f) for t, f in zip(_lt_total_for_share, _lt_foreign_for_share)]
panel["domestic_share_equity"] = [_domestic_share(t, f) for t, f in zip(panel["eq_mn"], panel["foreign_eq_mn"])]

# Domestic share must be in [0, 1]. Any violation indicates wrong definitions/data inputs.
_bad_parts = []
for _col in ["domestic_share_st_debt", "domestic_share_lt_debt", "domestic_share_equity"]:
    _s = pd.to_numeric(panel[_col], errors="coerce")
    _bad = panel[_s.notna() & ((_s < 0.0) | (_s > 1.0))][["country", "debt_source", "equity_source"]].copy()
    if not _bad.empty:
        _bad["metric"] = _col
        _bad["value"] = _s.loc[_bad.index].astype(float)
        _bad_parts.append(_bad)
if _bad_parts:
    _bad_all = pd.concat(_bad_parts, ignore_index=True).sort_values(["metric", "country"])
    raise ValueError("Domestic-share out-of-range detected; review grouping/definition logic.\n" + _bad_all.to_string(index=False))

_reserve_foreign_st = pd.to_numeric(panel["foreign_st_total_mn"], errors="coerce") if (USE_LOCAL_FOREIGN_DEBT and "foreign_st_total_mn" in panel.columns) else pd.to_numeric(panel["foreign_st_mn"], errors="coerce")
_reserve_foreign_lt = pd.to_numeric(panel["foreign_lt_total_mn"], errors="coerce") if (USE_LOCAL_FOREIGN_DEBT and "foreign_lt_total_mn" in panel.columns) else pd.to_numeric(panel["foreign_lt_mn"], errors="coerce")
panel["share_in_reserves_st_debt"] = [_share_in_reserves(r, f) for r, f in zip(panel["reserve_st_mn"], _reserve_foreign_st)]
panel["share_in_reserves_lt_debt"] = [_share_in_reserves(r, f) for r, f in zip(panel["reserve_lt_mn"], _reserve_foreign_lt)]
panel["share_in_reserves_equity"] = [_share_in_reserves(r, f) for r, f in zip(panel["reserve_eq_mn"], panel["foreign_eq_mn"])]

full_country_panel = panel.sort_values(["country", "year"]).reset_index(drop=True)

asset_rows = []
for _, r in full_country_panel.iterrows():
    asset_rows.extend(
        [
            {
                "country": r["country"],
                "year": int(r["year"]),
                "asset_class": "Short-term debt",
                "value_billion_usd": r["st_debt_billion_usd"],
                "domestic_share": r["domestic_share_st_debt"],
                "share_in_reserves": r["share_in_reserves_st_debt"],
            },
            {
                "country": r["country"],
                "year": int(r["year"]),
                "asset_class": "Long-term debt",
                "value_billion_usd": r["lt_debt_billion_usd"],
                "domestic_share": r["domestic_share_lt_debt"],
                "share_in_reserves": r["share_in_reserves_lt_debt"],
            },
            {
                "country": r["country"],
                "year": int(r["year"]),
                "asset_class": "Equity",
                "value_billion_usd": r["equity_billion_usd"],
                "domestic_share": r["domestic_share_equity"],
                "share_in_reserves": r["share_in_reserves_equity"],
            },
        ]
    )

full_country_panel_long = pd.DataFrame(asset_rows)
panel_2020 = full_country_panel[full_country_panel["year"] == TARGET_YEAR].copy()
panel_2020_long = full_country_panel_long[full_country_panel_long["year"] == TARGET_YEAR].copy()

print("Applied full-country harmonization for missing debt-foreign values.")

Applied full-country harmonization for missing debt-foreign values.


In [105]:
# Final presentation format aligned to the target table layout.
IMAGE_COUNTRY_ORDER = [
    "CAN", "USA",
    "AUT", "BEL", "FIN", "FRA", "DEU", "ITA", "NLD", "PRT", "ESP", "DNK", "ISR", "NOR", "SWE", "CHE", "GBR",
    "AUS", "HKG", "JPN", "NZL", "SGP",
    "GRC", "BRA", "CHN", "COL", "CZE", "HUN", "IND", "MYS", "MEX", "PHL", "POL", "RUS", "ZAF", "KOR", "THA",
]

IMAGE_GROUP_BY_ISO3 = {
    "CAN": "Developed markets: North America", "USA": "Developed markets: North America",
    "AUT": "Developed markets: Europe", "BEL": "Developed markets: Europe", "FIN": "Developed markets: Europe", "FRA": "Developed markets: Europe",
    "DEU": "Developed markets: Europe", "ITA": "Developed markets: Europe", "NLD": "Developed markets: Europe", "PRT": "Developed markets: Europe",
    "ESP": "Developed markets: Europe", "DNK": "Developed markets: Europe", "ISR": "Developed markets: Europe", "NOR": "Developed markets: Europe",
    "SWE": "Developed markets: Europe", "CHE": "Developed markets: Europe", "GBR": "Developed markets: Europe",
    "AUS": "Developed markets: Pacific", "HKG": "Developed markets: Pacific", "JPN": "Developed markets: Pacific", "NZL": "Developed markets: Pacific", "SGP": "Developed markets: Pacific",
    "GRC": "Emerging markets", "BRA": "Emerging markets", "CHN": "Emerging markets", "COL": "Emerging markets", "CZE": "Emerging markets", "HUN": "Emerging markets",
    "IND": "Emerging markets", "MYS": "Emerging markets", "MEX": "Emerging markets", "PHL": "Emerging markets", "POL": "Emerging markets", "RUS": "Emerging markets",
    "ZAF": "Emerging markets", "KOR": "Emerging markets", "THA": "Emerging markets",
}

ISO3_TO_NAME_IMAGE = {
    "CAN": "Canada", "USA": "United States", "AUT": "Austria", "BEL": "Belgium", "FIN": "Finland", "FRA": "France",
    "DEU": "Germany", "ITA": "Italy", "NLD": "Netherlands", "PRT": "Portugal", "ESP": "Spain", "DNK": "Denmark",
    "ISR": "Israel", "NOR": "Norway", "SWE": "Sweden", "CHE": "Switzerland", "GBR": "United Kingdom",
    "AUS": "Australia", "HKG": "Hong Kong", "JPN": "Japan", "NZL": "New Zealand", "SGP": "Singapore",
    "GRC": "Greece", "BRA": "Brazil", "CHN": "China", "COL": "Colombia", "CZE": "Czech Republic", "HUN": "Hungary",
    "IND": "India", "MYS": "Malaysia", "MEX": "Mexico", "PHL": "Philippines", "POL": "Poland", "RUS": "Russia",
    "ZAF": "South Africa", "KOR": "South Korea", "THA": "Thailand",
}

p2020 = panel_2020_long[panel_2020_long["year"] == TARGET_YEAR].copy()
wide = p2020.pivot(index="country", columns="asset_class", values=["value_billion_usd", "domestic_share", "share_in_reserves"])


def _get_metric(df_wide: pd.DataFrame, metric: str, asset: str) -> pd.Series:
    key = (metric, asset)
    if key not in df_wide.columns:
        return pd.Series(index=df_wide.index, dtype="float64")
    return pd.to_numeric(df_wide[key], errors="coerce")


country_rows = pd.DataFrame(index=IMAGE_COUNTRY_ORDER)
country_rows["market_group"] = [IMAGE_GROUP_BY_ISO3.get(c) for c in country_rows.index]
country_rows["Issuer"] = [ISO3_TO_NAME_IMAGE.get(c, c) for c in country_rows.index]
country_rows["ST_Billion_USD"] = _get_metric(wide, "value_billion_usd", "Short-term debt").reindex(country_rows.index)
country_rows["ST_Domestic_share"] = _get_metric(wide, "domestic_share", "Short-term debt").reindex(country_rows.index)
country_rows["ST_Share_in_reserves"] = _get_metric(wide, "share_in_reserves", "Short-term debt").reindex(country_rows.index)
country_rows["LT_Billion_USD"] = _get_metric(wide, "value_billion_usd", "Long-term debt").reindex(country_rows.index)
country_rows["LT_Domestic_share"] = _get_metric(wide, "domestic_share", "Long-term debt").reindex(country_rows.index)
country_rows["LT_Share_in_reserves"] = _get_metric(wide, "share_in_reserves", "Long-term debt").reindex(country_rows.index)
country_rows["EQ_Billion_USD"] = _get_metric(wide, "value_billion_usd", "Equity").reindex(country_rows.index)
country_rows["EQ_Domestic_share"] = _get_metric(wide, "domestic_share", "Equity").reindex(country_rows.index)
country_rows["EQ_Share_in_reserves"] = _get_metric(wide, "share_in_reserves", "Equity").reindex(country_rows.index)

for c in ["ST_Billion_USD", "LT_Billion_USD", "EQ_Billion_USD"]:
    country_rows[c] = country_rows[c].round(0)
for c in ["ST_Domestic_share", "ST_Share_in_reserves", "LT_Domestic_share", "LT_Share_in_reserves", "EQ_Domestic_share", "EQ_Share_in_reserves"]:
    country_rows[c] = country_rows[c].round(2)

group_order = [
    "Developed markets: North America",
    "Developed markets: Europe",
    "Developed markets: Pacific",
    "Emerging markets",
]

rows = []
for g in group_order:
    rows.append(
        {
            "market_group": g,
            "Issuer": g,
            "ST_Billion_USD": pd.NA,
            "ST_Domestic_share": pd.NA,
            "ST_Share_in_reserves": pd.NA,
            "LT_Billion_USD": pd.NA,
            "LT_Domestic_share": pd.NA,
            "LT_Share_in_reserves": pd.NA,
            "EQ_Billion_USD": pd.NA,
            "EQ_Domestic_share": pd.NA,
            "EQ_Share_in_reserves": pd.NA,
        }
    )
    rows.extend(country_rows[country_rows["market_group"] == g].to_dict(orient="records"))

final_display_table = pd.DataFrame(rows)

render_cols = [
    ("", "Issuer"),
    ("Short-term debt", "Billion\nUS$"),
    ("Short-term debt", "Domestic\nshare"),
    ("Short-term debt", "Share in\nreserves"),
    ("Long-term debt", "Billion\nUS$"),
    ("Long-term debt", "Domestic\nshare"),
    ("Long-term debt", "Share in\nreserves"),
    ("Equity", "Billion\nUS$"),
    ("Equity", "Domestic\nshare"),
    ("Equity", "Share in\nreserves"),
]

render_df = final_display_table[[
    "Issuer",
    "ST_Billion_USD", "ST_Domestic_share", "ST_Share_in_reserves",
    "LT_Billion_USD", "LT_Domestic_share", "LT_Share_in_reserves",
    "EQ_Billion_USD", "EQ_Domestic_share", "EQ_Share_in_reserves",
]].copy()
render_df.columns = pd.MultiIndex.from_tuples(render_cols)


def _format_billions(v):
    if pd.isna(v):
        return ""
    return f"{int(round(float(v))):,}"


def _format_share(v):
    if pd.isna(v):
        return ""
    return f"{float(v):.2f}"


def _row_style(row: pd.Series):
    group_names = set(group_order)
    is_group = str(row[("", "Issuer")]) in group_names
    if is_group:
        return ["font-style: italic; font-weight: 500;"] * len(row)
    return [""] * len(row)


styled = (
    render_df.style
    .format(
        {
            ("Short-term debt", "Billion\nUS$"): _format_billions,
            ("Long-term debt", "Billion\nUS$"): _format_billions,
            ("Equity", "Billion\nUS$"): _format_billions,
            ("Short-term debt", "Domestic\nshare"): _format_share,
            ("Short-term debt", "Share in\nreserves"): _format_share,
            ("Long-term debt", "Domestic\nshare"): _format_share,
            ("Long-term debt", "Share in\nreserves"): _format_share,
            ("Equity", "Domestic\nshare"): _format_share,
            ("Equity", "Share in\nreserves"): _format_share,
        }
    )
    .apply(_row_style, axis=1)
    .hide(axis="index")
)

display(styled)


In [106]:
# Compare current table against provided reference values; flag cells with >10% relative deviation
import numpy as np

ref = {
    "Canada": [521, 0.89, 0.06, 2522, 0.77, 0.05, 6515, 0.87, 0.00],
    "United States": [5489, 0.92, 0.04, 41070, 0.84, 0.05, 55563, 0.87, 0.00],
    "Austria": [18, 0.15, 0.65, 559, 0.49, 0.08, 282, 0.79, 0.00],
    "Belgium": [121, 0.69, 0.11, 1152, 0.62, 0.06, 1739, 0.93, 0.00],
    "Finland": [69, 0.60, 0.16, 384, 0.43, 0.09, 678, 0.78, 0.00],
    "France": [628, 0.79, 0.12, 5065, 0.65, 0.07, 10614, 0.89, 0.00],
    "Germany": [321, 0.43, 0.34, 4342, 0.58, 0.13, 3672, 0.57, 0.00],
    "Italy": [168, 0.62, 0.01, 3749, 0.78, 0.00, 2384, 0.89, 0.00],
    "Netherlands": [67, 0.40, 0.25, 1235, 0.45, 0.08, 6057, 0.80, 0.00],
    "Portugal": [26, 0.60, 0.01, 336, 0.69, 0.00, 386, 0.92, 0.00],
    "Spain": [195, 0.65, 0.12, 2621, 0.66, 0.04, 2292, 0.88, 0.00],
    "Denmark": [48, 0.58, 0.29, 670, 0.75, 0.02, 1705, 0.85, 0.00],
    "Israel": [29, 0.90, 0.01, 509, 0.90, 0.00, 660, 0.86, 0.00],
    "Norway": [30, 0.60, 0.08, 224, 0.19, 0.11, 1038, 0.91, 0.00],
    "Sweden": [123, 0.55, 0.07, 428, 0.46, 0.05, 3019, 0.88, 0.00],
    "Switzerland": [115, 0.76, 0.16, 908, 0.88, 0.03, 3747, 0.75, 0.00],
    "United Kingdom": [341, 0.84, 0.04, 4163, 0.84, 0.04, 6666, 0.76, 0.00],
    "Australia": [202, 0.89, 0.04, 1824, 0.71, 0.04, 1763, 0.73, 0.00],
    "Hong Kong": [13, 0.40, 0.06, 55, 0.11, 0.02, 2454, 0.81, 0.00],
    "Japan": [1889, 0.74, 0.12, 13467, 0.96, 0.01, 12172, 0.87, 0.00],
    "New Zealand": [9, 0.71, 0.10, 125, 0.76, 0.02, 1030, 0.96, 0.00],
    "Singapore": [41, 0.04, 0.39, 225, 0.66, 0.03, 887, 0.82, 0.00],
    "Greece": [16, 0.71, 0.00, 134, 0.86, 0.01, 182, 0.91, 0.00],
    "Brazil": [399, 0.98, 0.00, 1434, 0.90, 0.00, 2399, 0.91, 0.00],
    "China": [455, 0.80, 0.01, 17359, 0.97, 0.00, 15002, 0.77, 0.00],
    "Colombia": [27, 0.99, 0.00, 185, 0.81, 0.00, 483, 0.99, 0.00],
    "Czech Republic": [28, 0.00, 0.00, 95, 0.54, 0.00, 106, 0.98, 0.00],
    "Hungary": [7, 0.99, 0.00, 107, 0.79, 0.00, 104, 0.89, 0.00],
    "India": [326, 0.98, 0.00, 2027, 0.97, 0.00, 2354, 0.80, 0.00],
    "Malaysia": [18, 0.83, 0.01, 342, 0.84, 0.00, 447, 0.89, 0.00],
    "Mexico": [94, 0.96, 0.00, 591, 0.78, 0.00, 1313, 0.94, 0.00],
    "Philippines": [17, 0.99, 0.00, 94, 0.78, 0.00, 261, 0.88, 0.00],
    "Poland": [42, 0.99, 0.00, 301, 0.77, 0.00, 255, 0.91, 0.00],
    "Russia": [9, 0.42, 0.10, 298, 0.69, 0.01, 3039, 0.96, 0.00],
    "South Africa": [45, 0.98, 0.01, 196, 0.75, 0.01, 1105, 0.91, 0.00],
    "South Korea": [309, 0.92, 0.03, 2103, 0.92, 0.02, 3147, 0.85, 0.00],
    "Thailand": [75, 0.98, 0.00, 338, 0.92, 0.00, 538, 0.86, 0.00],
}

metric_cols = [
    "ST_Billion_USD", "ST_Domestic_share", "ST_Share_in_reserves",
    "LT_Billion_USD", "LT_Domestic_share", "LT_Share_in_reserves",
    "EQ_Billion_USD", "EQ_Domestic_share", "EQ_Share_in_reserves",
]

cur = country_rows[["Issuer"] + metric_cols].copy()
cur = cur[cur["Issuer"].isin(ref.keys())].copy()

rows = []
for _, r in cur.iterrows():
    issuer = str(r["Issuer"])
    std_vals = ref[issuer]
    for i, col in enumerate(metric_cols):
        cur_v = pd.to_numeric(r[col], errors="coerce")
        std_v = float(std_vals[i])
        if pd.isna(cur_v):
            rel = np.nan
        elif std_v == 0.0:
            rel = 0.0 if abs(float(cur_v)) < 1e-12 else np.inf
        else:
            rel = abs(float(cur_v) - std_v) / abs(std_v)
        rows.append({
            "Issuer": issuer,
            "Metric": col,
            "Current": None if pd.isna(cur_v) else float(cur_v),
            "Reference": std_v,
            "RelDiff": rel,
            "Over10pct": bool(pd.notna(rel) and rel > 0.10),
        })

cmp_df = pd.DataFrame(rows)
flag_df = cmp_df[cmp_df["Over10pct"]].copy().sort_values(["Issuer", "Metric"])

print("=== 偏差 >10% 的数据格 ===")
if flag_df.empty:
    print("无")
else:
    print(flag_df[["Issuer", "Metric", "Current", "Reference", "RelDiff"]].to_string(index=False))

print("\n总计超阈值格数:", int(flag_df.shape[0]))
reference_compare_over10 = flag_df

=== 偏差 >10% 的数据格 ===
        Issuer               Metric  Current  Reference   RelDiff
     Australia EQ_Share_in_reserves     0.01       0.00       inf
     Australia       LT_Billion_USD  2285.00    1824.00  0.252741
     Australia    LT_Domestic_share     0.80       0.71  0.126761
     Australia LT_Share_in_reserves     0.09       0.04  1.250000
     Australia       ST_Billion_USD   130.00     202.00  0.356436
     Australia ST_Share_in_reserves     0.03       0.04  0.250000
       Austria    LT_Domestic_share     0.56       0.49  0.142857
       Austria LT_Share_in_reserves     0.07       0.08  0.125000
       Austria    ST_Domestic_share     0.78       0.15  4.200000
       Austria ST_Share_in_reserves     0.21       0.65  0.676923
       Belgium EQ_Share_in_reserves     0.01       0.00       inf
       Belgium    LT_Domestic_share     0.71       0.62  0.145161
       Belgium    ST_Domestic_share     0.83       0.69  0.202899
       Belgium ST_Share_in_reserves     0.13       0.11

In [107]:
# Compact view + export full flagged cells
from pathlib import Path

repo_root = Path.cwd().resolve().parent
out_path = repo_root / "data" / "compare_over10_cells.csv"
reference_compare_over10.to_csv(out_path, index=False)
print("over10 count:", reference_compare_over10.shape[0])
print("saved:", out_path)
print(reference_compare_over10[["Issuer", "Metric", "Current", "Reference", "RelDiff"]].head(50).to_string(index=False))

over10 count: 146
saved: D:\Git\p09_koijen_yogo_2020\data\compare_over10_cells.csv
        Issuer               Metric  Current  Reference   RelDiff
     Australia EQ_Share_in_reserves     0.01       0.00       inf
     Australia       LT_Billion_USD  2285.00    1824.00  0.252741
     Australia    LT_Domestic_share     0.80       0.71  0.126761
     Australia LT_Share_in_reserves     0.09       0.04  1.250000
     Australia       ST_Billion_USD   130.00     202.00  0.356436
     Australia ST_Share_in_reserves     0.03       0.04  0.250000
       Austria    LT_Domestic_share     0.56       0.49  0.142857
       Austria LT_Share_in_reserves     0.07       0.08  0.125000
       Austria    ST_Domestic_share     0.78       0.15  4.200000
       Austria ST_Share_in_reserves     0.21       0.65  0.676923
       Belgium EQ_Share_in_reserves     0.01       0.00       inf
       Belgium    LT_Domestic_share     0.71       0.62  0.145161
       Belgium    ST_Domestic_share     0.83       0.69  0.

In [108]:
# Mark current table cells by deviation level
# >40% => !! ; >10% => !
marker_map = {}
for _, rr in cmp_df.iterrows():
    rel = rr["RelDiff"]
    if pd.isna(rel):
        continue
    key = (rr["Issuer"], rr["Metric"])
    if rel > 0.40:
        marker_map[key] = "!!"
    elif rel > 0.10:
        marker_map[key] = "!"

marked = country_rows[["Issuer"] + metric_cols].copy()
for col in metric_cols:
    marked[col] = marked.apply(
        lambda r: f"{r[col]} {marker_map[(r['Issuer'], col)]}" if (r["Issuer"], col) in marker_map else r[col],
        axis=1,
    )

print("已在当前表中标注：>40% 为 !!，>10% 为 !")
display(marked)
marked_over10_table = marked

已在当前表中标注：>40% 为 !!，>10% 为 !


,Issuer,ST_Billion_USD,ST_Domestic_share,ST_Share_in_reserves,LT_Billion_USD,LT_Domestic_share,LT_Share_in_reserves,EQ_Billion_USD,EQ_Domestic_share,EQ_Share_in_reserves
CAN,Canada,521.0,0.95,0.0 !!,2467.0,0.79,0.03 !!,6576.0,0.87,0.01 !!
USA,United States,5497.0,0.98,0.1 !!,38725.0,0.92,0.13 !!,55635.0,0.89,0.04 !!
AUT,Austria,17.0,0.78 !!,0.21 !!,560.0,0.56 !,0.07 !,286.0,0.78,0.0
BEL,Belgium,124.0,0.83 !,0.13 !,1150.0,0.71 !,0.06,1737.0,0.93,0.01 !!
FIN,Finland,65.0,0.87 !!,0.29 !!,398.0,0.61 !!,0.09,657.0,0.76,0.01 !!
FRA,France,628.0,0.92 !,0.04 !!,5084.0,0.74 !,0.08 !,10411.0,0.89,0.02 !!
DEU,Germany,310.0,0.83 !!,0.07 !!,4511.0,0.83 !!,0.07 !!,3696.0,0.73 !,0.02 !!
ITA,Italy,170.0,0.66,0.21 !!,3752.0,0.83,0.09 !!,2206.0,0.89,0.01 !!
NLD,Netherlands,72.0,0.54 !,0.05 !!,1235.0,0.24 !!,0.15 !!,6204.0,0.88,0.01 !!
PRT,Portugal,26.0,0.82 !,0.18 !!,343.0,0.74,0.14 !!,386.0,0.92,0.01 !!


In [109]:
# BIS-country audit against Appendix C rules (latest pulled BIS data)

bis_countries = [c for c in COUNTRY_ORDER if _debt_source(c, TARGET_YEAR) == "BIS"]



rows = []

for iso3 in bis_countries:

    iso2 = ISO3_TO_ISO2.get(iso3)

    if iso2 is None:

        continue



    dom_st, dom_lt = _bis_domestic_st_lt_usd_million(iso2, TARGET_YEAR)

    ids_st, ids_lt = _bis_ids_all_st_lt_usd_million(iso2, TARGET_YEAR)

    tot_st, tot_lt = _bis_total_st_lt_usd_million(iso2, TARGET_YEAR)

    fin_st, fin_lt = _bis_debt_st_lt_usd_million(iso3, TARGET_YEAR)



    dom_st_v = float(dom_st or 0.0)

    dom_lt_v = float(dom_lt or 0.0)

    ids_st_v = float(ids_st or 0.0)

    ids_lt_v = float(ids_lt or 0.0)

    fin_st_v = float(fin_st or 0.0)

    fin_lt_v = float(fin_lt or 0.0)

    tot_all = float(tot_st or 0.0) + float(tot_lt or 0.0)



    panel_row = panel[(panel["country"] == iso3) & (panel["year"].astype(str) == str(TARGET_YEAR))]

    panel_st_res = float(pd.to_numeric(panel_row["st_mn_residency"], errors="coerce").iloc[0]) if not panel_row.empty and pd.notna(pd.to_numeric(panel_row["st_mn_residency"], errors="coerce").iloc[0]) else 0.0

    panel_lt_res = float(pd.to_numeric(panel_row["lt_mn_residency"], errors="coerce").iloc[0]) if not panel_row.empty and pd.notna(pd.to_numeric(panel_row["lt_mn_residency"], errors="coerce").iloc[0]) else 0.0

    panel_st_total = float(pd.to_numeric(panel_row["st_mn"], errors="coerce").iloc[0]) if not panel_row.empty and pd.notna(pd.to_numeric(panel_row["st_mn"], errors="coerce").iloc[0]) else float("nan")

    panel_lt_total = float(pd.to_numeric(panel_row["lt_mn"], errors="coerce").iloc[0]) if not panel_row.empty and pd.notna(pd.to_numeric(panel_row["lt_mn"], errors="coerce").iloc[0]) else float("nan")

    panel_st_local = float(pd.to_numeric(panel_row["st_local_mn"], errors="coerce").iloc[0]) if not panel_row.empty and pd.notna(pd.to_numeric(panel_row["st_local_mn"], errors="coerce").iloc[0]) else float("nan")

    panel_lt_local = float(pd.to_numeric(panel_row["lt_local_mn"], errors="coerce").iloc[0]) if not panel_row.empty and pd.notna(pd.to_numeric(panel_row["lt_local_mn"], errors="coerce").iloc[0]) else float("nan")



    rows.append({

        "country": iso3,

        "dom_st": dom_st_v,

        "dom_lt": dom_lt_v,

        "ids_st": ids_st_v,

        "ids_lt": ids_lt_v,

        "tot_all": tot_all,

        "final_st": fin_st_v,

        "final_lt": fin_lt_v,

        "sum_dom_ids_st": dom_st_v + ids_st_v,

        "sum_dom_ids_lt": dom_lt_v + ids_lt_v,

        "panel_st_res": panel_st_res,

        "panel_lt_res": panel_lt_res,

        "panel_st_total": panel_st_total,

        "panel_lt_total": panel_lt_total,

        "panel_st_local": panel_st_local,

        "panel_lt_local": panel_lt_local,

    })



bis_audit = pd.DataFrame(rows)

if bis_audit.empty:

    print("No BIS-source countries in current target year.")

else:

    # Core identity checks (Appendix C: debt = domestic + IDS)

    bis_audit["gap_st_final_vs_domids"] = bis_audit["final_st"] - bis_audit["sum_dom_ids_st"]

    bis_audit["gap_lt_final_vs_domids"] = bis_audit["final_lt"] - bis_audit["sum_dom_ids_lt"]

    bis_audit["gap_st_panel_vs_final"] = bis_audit["panel_st_res"] - bis_audit["final_st"]

    bis_audit["gap_lt_panel_vs_final"] = bis_audit["panel_lt_res"] - bis_audit["final_lt"]

    bis_audit["flag_local_gt_total_st"] = bis_audit["panel_st_local"] > bis_audit["panel_st_total"] + 1e-9

    bis_audit["flag_local_gt_total_lt"] = bis_audit["panel_lt_local"] > bis_audit["panel_lt_total"] + 1e-9



    tol = 1e-6

    failed = bis_audit[

        (bis_audit["gap_st_panel_vs_final"].abs() > tol)

        | (bis_audit["gap_lt_panel_vs_final"].abs() > tol)

        | (bis_audit["flag_local_gt_total_st"])

        | (bis_audit["flag_local_gt_total_lt"])

    ].copy()



    print("BIS countries checked:", bis_audit.shape[0])

    print("Identity failures (panel vs final or local>total):", failed.shape[0])

    if failed.empty:

        print("All BIS-country short/long debt stock calculations are aligned with current Appendix C implementation.")

    else:

        show_cols = [

            "country",

            "panel_st_res", "final_st", "gap_st_panel_vs_final",

            "panel_lt_res", "final_lt", "gap_lt_panel_vs_final",

            "panel_st_total", "panel_lt_total",

            "panel_st_local", "panel_lt_local",

            "flag_local_gt_total_st", "flag_local_gt_total_lt",

        ]

        display(failed[show_cols].sort_values("country"))



    display(

        bis_audit[[

            "country", "dom_st", "ids_st", "final_st", "panel_st_res", "panel_st_total", "panel_st_local",

            "dom_lt", "ids_lt", "final_lt", "panel_lt_res", "panel_lt_total", "panel_lt_local"

        ]].sort_values("country")

    )


BIS countries checked: 11
Identity failures (panel vs final or local>total): 0
All BIS-country short/long debt stock calculations are aligned with current Appendix C implementation.


,country,dom_st,ids_st,final_st,panel_st_res,panel_st_total,panel_st_local,dom_lt,ids_lt,final_lt,panel_lt_res,panel_lt_total,panel_lt_local
0,AUS,3.022197e+05,29180.000000,129353.615582,129353.615582,129718.015744,129718.015744,2.195582e+06,534282.000000,2.368448e+06,2.368448e+06,2.284628e+06,2.284628e+06
4,CHN,1.988278e+06,4680.000000,365694.817376,365694.817376,508009.250531,508009.250531,1.656730e+07,232786.000000,1.818988e+07,1.818988e+07,1.825430e+07,1.825430e+07
1,HKG,1.345296e+05,73996.000000,208525.591011,208525.591011,47888.441857,47888.441857,2.054086e+04,284903.000000,3.054439e+05,3.054439e+05,8.980871e+04,8.980871e+04
5,IND,1.153722e+05,48.238391,115420.451563,115420.451563,122620.518472,122620.518472,1.450169e+06,67956.761609,1.518126e+06,1.518126e+06,1.573156e+06,1.573156e+06
6,MYS,1.751614e+04,400.000000,17916.144548,17916.144548,18719.919842,18719.919842,3.897326e+05,51927.000000,4.416596e+05,4.416596e+05,4.453839e+05,4.453839e+05
2,NZL,5.183964e+03,2884.000000,8067.963886,8067.963886,5257.029776,5257.029776,9.574461e+04,26527.000000,1.222716e+05,1.222716e+05,6.675080e+04,6.675080e+04
7,PHL,4.563251e+03,256.937196,4820.188488,4820.188488,4742.277112,4742.277112,2.447266e+04,67100.062804,9.157273e+04,9.157273e+04,8.503417e+04,8.503417e+04
8,RUS,1.859483e+04,10530.743323,44065.423213,44065.423213,43296.206629,43296.206629,3.926450e+05,87747.256677,3.671745e+05,3.671745e+05,3.693183e+05,3.693183e+05
3,SGP,1.833030e+05,23809.000000,69622.504931,69622.504931,61887.552133,61887.552133,3.594537e+05,161799.000000,4.731342e+05,4.731342e+05,5.192388e+05,5.192388e+05
10,THA,8.183043e+04,262.872375,5974.228265,5974.228265,10128.052210,10128.052210,4.074987e+05,21268.127625,4.833549e+05,4.833549e+05,4.527720e+05,4.527720e+05


In [110]:
# India focused BIS debug: identify where short/long split may be lost

iso3 = "IND"

iso2 = ISO3_TO_ISO2[iso3]

year = TARGET_YEAR



dom_st, dom_lt = _bis_domestic_st_lt_usd_million(iso2, year)

ids_st, ids_lt = _bis_ids_all_st_lt_usd_million(iso2, year)

tot_st, tot_lt = _bis_total_st_lt_usd_million(iso2, year)

fin_st, fin_lt = _bis_debt_st_lt_usd_million(iso3, year)



print("IND core (mn USD)")

print({

    "dom_st": dom_st,

    "dom_lt": dom_lt,

    "ids_st": ids_st,

    "ids_lt": ids_lt,

    "tot_st": tot_st,

    "tot_lt": tot_lt,

    "final_st": fin_st,

    "final_lt": fin_lt,

})



# Inspect domestic BIS maturity availability at 2020Q4 and 2020 annual

x = _bis_base(iso2)

x = x[x["TIME_PERIOD"].astype(str).isin([f"{year}-Q4", str(year)])].copy()

x = _bis_pick_currency_slice(x)

x = _bis_pick_valuation_slice(x)

x["OBS_VALUE"] = pd.to_numeric(x["OBS_VALUE"], errors="coerce")



print("\nIND domestic maturity by REF_SECTOR/time")

dom_diag = (

    x.groupby(["TIME_PERIOD", "REF_SECTOR", "MATURITY"], as_index=False)["OBS_VALUE"]

    .sum(min_count=1)

    .sort_values(["TIME_PERIOD", "REF_SECTOR", "MATURITY"])

)

display(dom_diag)



print("\nIND IDS maturity split @Q4")

ids = pd.read_parquet(BIS_IDS_ALL_PATH).copy()

for c in ["ISSUER_RES", "ISSUE_OR_MAT", "TIME_PERIOD", "ISSUE_CUR_GROUP", "ISSUER_BUS_IMM"]:

    if c in ids.columns:

        ids[c] = ids[c].astype(str)

ids = ids[(ids["ISSUER_RES"] == iso2) & (ids["TIME_PERIOD"] == f"{year}-Q4") & (ids["ISSUE_CUR_GROUP"] == "A")].copy()

if "ISSUER_BUS_IMM" in ids.columns:

    ids_agg = ids[ids["ISSUER_BUS_IMM"] == "1"].copy()

    if not ids_agg.empty:

        ids = ids_agg

if "million_usd" in ids.columns:

    ids["million_usd"] = pd.to_numeric(ids["million_usd"], errors="coerce")

else:

    ids["million_usd"] = (pd.to_numeric(ids.get("OBS_VALUE"), errors="coerce") * (10 ** pd.to_numeric(ids.get("UNIT_MULT"), errors="coerce").fillna(0))) / 1e6

ids_diag = ids.groupby("ISSUE_OR_MAT", as_index=False)["million_usd"].sum(min_count=1).sort_values("ISSUE_OR_MAT")

display(ids_diag)



panel_ind = panel[(panel["country"] == iso3) & (panel["year"].astype(str) == str(year))].copy()

print("\nIND panel values (mn USD)")

display(panel_ind[["country", "year", "st_mn_residency", "lt_mn_residency", "st_mn", "lt_mn", "st_local_mn", "lt_local_mn"]])


IND core (mn USD)
{'dom_st': 115372.213172, 'dom_lt': 1450169.28096, 'ids_st': 48.238391444317124, 'ids_lt': 67956.76160855568, 'tot_st': 749.204014549026, 'tot_lt': 1055455.563265451, 'final_st': 115420.45156344432, 'final_lt': 1518126.0425685556}

IND domestic maturity by REF_SECTOR/time


,TIME_PERIOD,REF_SECTOR,MATURITY,OBS_VALUE
0,2020-Q4,S13,L,1450.169281
1,2020-Q4,S13,S,115.372213
2,2020-Q4,S13,T,1056.204767



IND IDS maturity split @Q4


,ISSUE_OR_MAT,million_usd
0,A,68005.0
1,K,68005.0



IND panel values (mn USD)


,country,year,st_mn_residency,lt_mn_residency,st_mn,lt_mn,st_local_mn,lt_local_mn
28,IND,2020,115420.451563,1.518126e+06,122620.518472,1.573156e+06,122620.518472,1.573156e+06


In [111]:
# India domestic currency coverage diagnostic

iso2 = "IN"

q = _bis_base(iso2)

q = q[(q["FREQ"].astype(str) == "Q") & (q["TIME_PERIOD"].astype(str) == f"{TARGET_YEAR}-Q4")].copy()

q["OBS_VALUE"] = pd.to_numeric(q["OBS_VALUE"], errors="coerce")

print("IND 2020Q4 domestic by currency:")

ccy_diag = q.groupby(["CURRENCY_DENOM", "MATURITY"], as_index=False)["OBS_VALUE"].sum(min_count=1)

display(ccy_diag.sort_values(["CURRENCY_DENOM", "MATURITY"]))

print("Currency totals:")

display(q.groupby("CURRENCY_DENOM", as_index=False)["OBS_VALUE"].sum(min_count=1).sort_values("OBS_VALUE", ascending=False))


IND 2020Q4 domestic by currency:


,CURRENCY_DENOM,MATURITY,OBS_VALUE
0,X1,T,0.000000
1,XDC,T,1056.204767
2,_T,L,1450.169281
3,_T,S,115.372213
4,_T,T,1056.204767


Currency totals:


,CURRENCY_DENOM,OBS_VALUE
2,_T,2621.746261
1,XDC,1056.204767
0,X1,0.000000


In [112]:
# India REF_SECTOR coverage check (including S1M)

x = bis[

    (bis["REF_AREA"].astype(str) == "IN")

    & (bis["COUNTERPART_SECTOR"].astype(str) == "S1")

    & (bis["CONSOLIDATION"].astype(str) == "N")

    & (bis["ACCOUNTING_ENTRY"].astype(str) == "L")

    & (bis["STO"].astype(str) == "LE")

    & (bis["INSTR_ASSET"].astype(str) == "F3")

    & (bis["TRANSFORMATION"].astype(str) == "N")

    & (bis["UNIT_MEASURE"].astype(str) == "USD")

    & (bis["PRICES"].astype(str) == "V")

    & (bis["EXPENDITURE"].astype(str) == "_Z")

    & (bis["FREQ"].astype(str) == "Q")

    & (bis["TIME_PERIOD"].astype(str) == f"{TARGET_YEAR}-Q4")

][["REF_SECTOR", "CURRENCY_DENOM", "MATURITY", "OBS_VALUE"]].copy()

x["OBS_VALUE"] = pd.to_numeric(x["OBS_VALUE"], errors="coerce")

print("IND REF_SECTOR coverage")

display(x.groupby(["REF_SECTOR", "CURRENCY_DENOM", "MATURITY"], as_index=False)["OBS_VALUE"].sum(min_count=1).sort_values(["REF_SECTOR", "CURRENCY_DENOM", "MATURITY"]))


IND REF_SECTOR coverage


,REF_SECTOR,CURRENCY_DENOM,MATURITY,OBS_VALUE
0,S13,X1,T,0.000000
1,S13,XDC,T,1056.204767
2,S13,_T,L,1450.169281
3,S13,_T,S,115.372213
4,S13,_T,T,1056.204767


In [118]:
# IDS widened-coverage audit: which currency-group path is effectively used per issuer-year
# Output: data/ids_selection_path_audit.csv

from pathlib import Path


def _ids_apply_preference_local(ids_df: pd.DataFrame) -> pd.DataFrame:
    ids_df = ids_df.copy()
    if "ISSUE_CUR" in ids_df.columns and (ids_df["ISSUE_CUR"].astype(str) == "TO1").any():
        ids_df = ids_df[ids_df["ISSUE_CUR"].astype(str) == "TO1"].copy()
    if "ISSUE_RE_MAT" in ids_df.columns and (ids_df["ISSUE_RE_MAT"].astype(str) == "A").any():
        ids_df = ids_df[ids_df["ISSUE_RE_MAT"].astype(str) == "A"].copy()
    for _c in ["ISSUE_RATE", "ISSUE_RISK", "ISSUE_COL"]:
        if _c in ids_df.columns and (ids_df[_c].astype(str) == "A").any():
            ids_df = ids_df[ids_df[_c].astype(str) == "A"].copy()
    if "ISSUER_BUS_IMM" in ids_df.columns:
        ids_agg = ids_df[ids_df["ISSUER_BUS_IMM"].astype(str) == "1"].copy()
        if not ids_agg.empty:
            ids_df = ids_agg
    return ids_df


def _ids_pick_group_local(ids_df: pd.DataFrame) -> pd.DataFrame:
    if ids_df.empty or "ISSUE_CUR_GROUP" not in ids_df.columns:
        return ids_df
    grp = ids_df["ISSUE_CUR_GROUP"].astype(str)
    if (grp == "A").any():
        return ids_df[grp == "A"].copy()
    has_d = (grp == "D").any()
    has_f = (grp == "F").any()
    if has_d and has_f:
        return ids_df[grp.isin(["D", "F"])].copy()
    if has_d:
        return ids_df[grp == "D"].copy()
    if has_f:
        return ids_df[grp == "F"].copy()
    return ids_df


def _sum_million(ids_df: pd.DataFrame, mat: str) -> float:
    if ids_df.empty:
        return 0.0
    if "million_usd" in ids_df.columns:
        vals = pd.to_numeric(ids_df.loc[ids_df["ISSUE_OR_MAT"].astype(str) == mat, "million_usd"], errors="coerce")
        return float(vals.sum())
    vals = ids_df[ids_df["ISSUE_OR_MAT"].astype(str) == mat]
    out = (
        pd.to_numeric(vals.get("OBS_VALUE"), errors="coerce")
        * (10 ** pd.to_numeric(vals.get("UNIT_MULT"), errors="coerce").fillna(0))
    ) / 1e6
    return float(out.sum())


audit_rows = []
ids_all = pd.read_parquet(BIS_IDS_ALL_PATH).copy() if Path(BIS_IDS_ALL_PATH).exists() else pd.DataFrame()

if not ids_all.empty:
    for c in ["ISSUER_RES", "TIME_PERIOD", "ISSUE_OR_MAT", "ISSUE_CUR_GROUP", "MEASURE", "MARKET", "ISSUE_TYPE", "ISSUER_BUS_IMM"]:
        if c in ids_all.columns:
            ids_all[c] = ids_all[c].astype(str)

for iso3 in COUNTRY_ORDER:
    iso2 = ISO3_TO_ISO2.get(iso3)
    year = int(TARGET_YEAR)

    row = {
        "country_iso3": iso3,
        "country_iso2": iso2,
        "year": year,
        "path_group": "none",
        "split_source": "none",
        "has_A": False,
        "has_D": False,
        "has_F": False,
        "A_total_mn": 0.0,
        "A_st_mn": 0.0,
        "A_lt_mn": 0.0,
        "DF_total_mn": 0.0,
        "DF_st_mn": 0.0,
        "DF_lt_mn": 0.0,
        "df_split_used_as_fallback": False,
    }

    if iso2 is None or ids_all.empty:
        audit_rows.append(row)
        continue

    x = ids_all[
        (ids_all.get("ISSUER_RES", "").astype(str) == str(iso2))
        & (ids_all.get("TIME_PERIOD", "").astype(str) == f"{year}-Q4")
        & (ids_all.get("MEASURE", "").astype(str) == "I")
        & (ids_all.get("MARKET", "").astype(str) == "C")
        & (ids_all.get("ISSUE_TYPE", "").astype(str) == "A")
        & (ids_all.get("ISSUE_OR_MAT", "").astype(str).isin(["A", "C", "K"]))
    ].copy()

    if x.empty:
        audit_rows.append(row)
        continue

    pref_fn = globals().get("_bis_ids_apply_preference")
    sel_fn = globals().get("_bis_ids_select_currency_group")

    x_pref = pref_fn(x) if callable(pref_fn) else _ids_apply_preference_local(x)
    x_sel = sel_fn(x_pref) if callable(sel_fn) else _ids_pick_group_local(x_pref)

    grp_all = x_pref["ISSUE_CUR_GROUP"].astype(str)
    row["has_A"] = bool((grp_all == "A").any())
    row["has_D"] = bool((grp_all == "D").any())
    row["has_F"] = bool((grp_all == "F").any())

    x_a = x_pref[x_pref["ISSUE_CUR_GROUP"].astype(str) == "A"].copy()
    x_df = x_pref[x_pref["ISSUE_CUR_GROUP"].astype(str).isin(["D", "F"])].copy()

    row["A_total_mn"] = _sum_million(x_a, "A")
    row["A_st_mn"] = _sum_million(x_a, "C")
    row["A_lt_mn"] = _sum_million(x_a, "K")
    row["DF_total_mn"] = _sum_million(x_df, "A")
    row["DF_st_mn"] = _sum_million(x_df, "C")
    row["DF_lt_mn"] = _sum_million(x_df, "K")

    used_groups = sorted(set(x_sel["ISSUE_CUR_GROUP"].astype(str).tolist())) if (not x_sel.empty and "ISSUE_CUR_GROUP" in x_sel.columns) else []
    if used_groups == ["A"]:
        row["path_group"] = "A"
    elif used_groups == ["D"]:
        row["path_group"] = "D"
    elif used_groups == ["F"]:
        row["path_group"] = "F"
    elif used_groups == ["D", "F"]:
        row["path_group"] = "D+F"
    elif used_groups:
        row["path_group"] = "+".join(used_groups)

    # Detect if widened D/F split could be (or is) used when A total exists but A C/K split is missing.
    a_split_missing = row["A_total_mn"] > 0.0 and (row["A_st_mn"] <= 0.0 or row["A_lt_mn"] <= 0.0)
    df_split_available = row["DF_st_mn"] > 0.0 and row["DF_lt_mn"] > 0.0

    if row["path_group"] in {"D", "F", "D+F"}:
        row["split_source"] = "DF"
    elif row["path_group"] == "A":
        if a_split_missing and df_split_available:
            row["split_source"] = "A_total + DF_split_fallback"
            row["df_split_used_as_fallback"] = True
        elif row["A_st_mn"] > 0.0 and row["A_lt_mn"] > 0.0:
            row["split_source"] = "A_direct_split"
        elif row["A_total_mn"] > 0.0:
            row["split_source"] = "A_total + nearest/global_ratio"
        else:
            row["split_source"] = "A_but_no_amount"

    audit_rows.append(row)

ids_audit = pd.DataFrame(audit_rows)
out_audit = project_root / "data" / "ids_selection_path_audit.csv"
ids_audit.to_csv(out_audit, index=False)

print("Wrote:", out_audit)
print("Path group counts:")
print(ids_audit["path_group"].value_counts(dropna=False).to_string())
print("\nSplit source counts:")
print(ids_audit["split_source"].value_counts(dropna=False).to_string())
print("\nRows with DF split fallback:", int(ids_audit["df_split_used_as_fallback"].sum()))

display(ids_audit[[
    "country_iso3", "path_group", "split_source", "has_A", "has_D", "has_F",
    "A_total_mn", "A_st_mn", "A_lt_mn", "DF_total_mn", "DF_st_mn", "DF_lt_mn", "df_split_used_as_fallback"
]].sort_values(["path_group", "country_iso3"]))

Wrote: D:\Git\p09_koijen_yogo_2020\data\ids_selection_path_audit.csv
Path group counts:
path_group
A    37

Split source counts:
split_source
A_direct_split                    31
A_total + nearest/global_ratio     6

Rows with DF split fallback: 0


,country_iso3,path_group,split_source,has_A,has_D,has_F,A_total_mn,A_st_mn,A_lt_mn,DF_total_mn,DF_st_mn,DF_lt_mn,df_split_used_as_fallback
17,AUS,A,A_direct_split,True,True,True,563462.0,29180.0,534282.0,563462.0,29180.0,534281.0,False
2,AUT,A,A_direct_split,True,True,True,330232.0,24977.0,305255.0,330232.0,24976.0,305255.0,False
3,BEL,A,A_direct_split,True,True,True,189091.0,66.0,189024.0,189091.0,66.0,189025.0,False
22,BRA,A,A_direct_split,True,True,True,112069.0,361.0,111708.0,112069.0,361.0,111708.0,False
0,CAN,A,A_direct_split,True,True,True,993747.0,9426.0,984321.0,993747.0,9425.0,984322.0,False
15,CHE,A,A_direct_split,True,True,True,116764.0,49.0,116715.0,116764.0,49.0,116715.0,False
23,CHN,A,A_direct_split,True,True,True,237466.0,4680.0,232786.0,237466.0,4680.0,232786.0,False
24,COL,A,A_direct_split,True,True,True,62962.0,80.0,62882.0,62963.0,80.0,62883.0,False
25,CZE,A,A_direct_split,True,True,True,35259.0,41.0,35217.0,35259.0,41.0,35218.0,False
7,DEU,A,A_direct_split,True,True,True,1361998.0,120492.0,1241507.0,1361998.0,120492.0,1241506.0,False


In [ ]:
# Strict logic-effect audit: check whether core Appendix-C fallback rules actually trigger,# and flag likely mis-identification cases.from pathlib import Pathimport pandas as pdimport numpy as npdef _diag_bis_domestic_logic_for_country(iso3: str, year: int) -> dict:    iso2 = ISO3_TO_ISO2.get(str(iso3))    row = {        "country_iso3": str(iso3),        "country_iso2": iso2,        "year": int(year),        "debt_source": _debt_source(str(iso3), int(year)),        "dom_st": np.nan,        "dom_lt": np.nan,        "dom_all_obs": np.nan,        "ids_st": np.nan,        "ids_lt": np.nan,        "ids_all": np.nan,        "tot_st": np.nan,        "tot_lt": np.nan,        "tot_all": np.nan,        "dom_raw_split_ok": False,        "dom_equals_total": False,        "dom_missing_basic": False,        "dom_split_unreliable": False,        "dom_inconsistent_with_identity": False,        "dom_overshoots_total": False,        "use_reconstructed_domestic": False,        "dom_recon_all_raw": np.nan,        "dom_tol": np.nan,        "suspicious_negative_recon": False,        "suspicious_dom_exists_but_forced_recon": False,        "suspicious_large_identity_gap_no_recon": False,    }    if iso2 is None:        return row    ids_st_mn, ids_lt_mn = _bis_ids_all_st_lt_usd_million(iso2, int(year))    dom_st_mn, dom_lt_mn = _bis_domestic_st_lt_usd_million(iso2, int(year))    tot_st_mn, tot_lt_mn = _bis_total_st_lt_usd_million(iso2, int(year))    dom_raw_split_ok = _bis_has_raw_st_lt_split(iso2, int(year))    ids_all = float(ids_st_mn) + float(ids_lt_mn)    dom_all_obs = float(dom_st_mn or 0.0) + float(dom_lt_mn or 0.0)    tot_all = float(tot_st_mn) + float(tot_lt_mn)    dom_equals_total = (tot_all > 0.0) and (abs(dom_all_obs - tot_all) <= 1e-6 * max(1.0, abs(tot_all)))    dom_missing_basic = (dom_st_mn is None or dom_lt_mn is None) or (dom_all_obs <= 0.0)    dom_split_still_missing = (dom_all_obs > 0.0) and ((float(dom_st_mn or 0.0) <= 0.0) or (float(dom_lt_mn or 0.0) <= 0.0))    dom_split_unreliable = (not dom_raw_split_ok) and dom_split_still_missing    dom_recon_all_raw = max(0.0, tot_all - ids_all) if tot_all > 0.0 else 0.0    dom_tol = max(0.20 * max(1.0, abs(tot_all)), 250.0)    ids_total_conflict = (tot_all > 0.0) and (ids_all > 1.15 * tot_all)    ids_material = (tot_all > 0.0) and (ids_all > max(0.05 * tot_all, 100.0))    dom_inconsistent_with_identity = (        (tot_all > 0.0)        and (not ids_total_conflict)        and ids_material        and (abs(dom_all_obs - dom_recon_all_raw) > max(dom_tol, 0.35 * max(1.0, abs(dom_recon_all_raw))))    )    dom_equals_total_suspicious = dom_equals_total and ids_material and (not ids_total_conflict)    dom_overshoots_total = (tot_all > 0.0) and (dom_all_obs > (tot_all + max(0.10 * max(1.0, abs(tot_all)), 250.0)))    use_reconstructed_domestic = (        dom_missing_basic        or dom_split_unreliable        or dom_overshoots_total        or dom_equals_total_suspicious    )    suspicious_negative_recon = (tot_all > 0.0) and (ids_all > tot_all + dom_tol)    suspicious_dom_exists_but_forced_recon = (        (dom_all_obs > 0.0)        and dom_raw_split_ok        and use_reconstructed_domestic        and (not dom_missing_basic)    )    suspicious_large_identity_gap_no_recon = (        (tot_all > 0.0)        and (not use_reconstructed_domestic)        and (abs(dom_all_obs - dom_recon_all_raw) > dom_tol)    )    row.update({        "dom_st": float(dom_st_mn) if dom_st_mn is not None else np.nan,        "dom_lt": float(dom_lt_mn) if dom_lt_mn is not None else np.nan,        "dom_all_obs": float(dom_all_obs),        "ids_st": float(ids_st_mn),        "ids_lt": float(ids_lt_mn),        "ids_all": float(ids_all),        "tot_st": float(tot_st_mn),        "tot_lt": float(tot_lt_mn),        "tot_all": float(tot_all),        "dom_raw_split_ok": bool(dom_raw_split_ok),        "dom_equals_total": bool(dom_equals_total),        "dom_missing_basic": bool(dom_missing_basic),        "dom_split_unreliable": bool(dom_split_unreliable),        "dom_inconsistent_with_identity": bool(dom_inconsistent_with_identity),        "dom_overshoots_total": bool(dom_overshoots_total),        "use_reconstructed_domestic": bool(use_reconstructed_domestic),        "dom_recon_all_raw": float(dom_recon_all_raw),        "dom_tol": float(dom_tol),        "suspicious_negative_recon": bool(suspicious_negative_recon),        "suspicious_dom_exists_but_forced_recon": bool(suspicious_dom_exists_but_forced_recon),        "suspicious_large_identity_gap_no_recon": bool(suspicious_large_identity_gap_no_recon),    })    return row# Scope 1: table year only (matches main output)rows_2020 = []for iso3 in COUNTRY_ORDER:    if _debt_source(iso3, int(TARGET_YEAR)) != "BIS":        continue    rows_2020.append(_diag_bis_domestic_logic_for_country(iso3, int(TARGET_YEAR)))diag_2020 = pd.DataFrame(rows_2020)# Scope 2: quick cross-year stress scan for BIS issuers in sample periodstress_rows = []for iso3 in COUNTRY_ORDER:    for yy in range(int(SAMPLE_START_YEAR.get(iso3, 2003)), int(TARGET_YEAR) + 1):        if _debt_source(iso3, yy) != "BIS":            continue        stress_rows.append(_diag_bis_domestic_logic_for_country(iso3, yy))diag_stress = pd.DataFrame(stress_rows)out_dir = project_root / "data"out_dir.mkdir(parents=True, exist_ok=True)p1 = out_dir / "bis_logic_strict_audit_2020.csv"p2 = out_dir / "bis_logic_strict_audit_stress.csv"diag_2020.to_csv(p1, index=False)diag_stress.to_csv(p2, index=False)# Summariesprint("Wrote:", p1)print("Wrote:", p2)print("\n=== 2020 trigger counts ===")for c in [    "dom_missing_basic", "dom_split_unreliable", "dom_inconsistent_with_identity",    "dom_overshoots_total", "use_reconstructed_domestic",    "suspicious_negative_recon", "suspicious_dom_exists_but_forced_recon",    "suspicious_large_identity_gap_no_recon",]:    if c in diag_2020.columns:        print(f"{c}: {int(diag_2020[c].sum())} / {len(diag_2020)}")print("\n=== stress trigger counts ===")for c in [    "dom_missing_basic", "dom_split_unreliable", "dom_inconsistent_with_identity",    "dom_overshoots_total", "use_reconstructed_domestic",    "suspicious_negative_recon", "suspicious_dom_exists_but_forced_recon",    "suspicious_large_identity_gap_no_recon",]:    if c in diag_stress.columns:        print(f"{c}: {int(diag_stress[c].sum())} / {len(diag_stress)}")print("\nTop suspicious (2020):")sus_2020 = diag_2020[    diag_2020[[        "suspicious_negative_recon",        "suspicious_dom_exists_but_forced_recon",        "suspicious_large_identity_gap_no_recon",    ]].any(axis=1)].copy()if sus_2020.empty:    print("None")else:    display(sus_2020[[        "country_iso3", "year", "dom_all_obs", "ids_all", "tot_all", "dom_recon_all_raw", "dom_tol",        "dom_missing_basic", "dom_split_unreliable", "dom_inconsistent_with_identity", "dom_overshoots_total",        "use_reconstructed_domestic",        "suspicious_negative_recon", "suspicious_dom_exists_but_forced_recon", "suspicious_large_identity_gap_no_recon",    ]].sort_values(["country_iso3", "year"]))print("\nTop suspicious (stress):")sus_stress = diag_stress[    diag_stress[[        "suspicious_negative_recon",        "suspicious_dom_exists_but_forced_recon",        "suspicious_large_identity_gap_no_recon",    ]].any(axis=1)].copy()if sus_stress.empty:    print("None")else:    display(sus_stress[[        "country_iso3", "year", "dom_all_obs", "ids_all", "tot_all", "dom_recon_all_raw", "dom_tol",        "dom_missing_basic", "dom_split_unreliable", "dom_inconsistent_with_identity", "dom_overshoots_total",        "use_reconstructed_domestic",        "suspicious_negative_recon", "suspicious_dom_exists_but_forced_recon", "suspicious_large_identity_gap_no_recon",    ]].sort_values(["country_iso3", "year"]))